In [2]:
pip install torch_geometric

  Obtaining dependency information for torch_geometric from https://files.pythonhosted.org/packages/03/9f/157e913626c1acfb3b19ce000b1a6e4e4fb177c0bc0ea0c67ca5bd714b5a/torch_geometric-2.6.1-py3-none-any.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 18.6 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [4]:
import rdflib
from rdflib import Graph, Namespace, URIRef, Literal, BNode
from rdflib.namespace import RDF, RDFS, XSD, OWL
import networkx as nx

In [5]:
nkb = Graph()
nkb.parse('./mappings/NKB_RDF_V3.ttl')

<Graph identifier=N1d8ca63c21c044148465839a5b38da9f (<class 'rdflib.graph.Graph'>)>

In [6]:
len(nkb)

1369675

In [13]:
print(nkb.namespaces)

<bound method Graph.namespaces of <Graph identifier=N1d8ca63c21c044148465839a5b38da9f (<class 'rdflib.graph.Graph'>)>>


In [14]:
import rdflib
from rdflib import Graph
import torch
import torch_geometric as pyg
import pandas as pd
import numpy as np

# Load your RDF file
g = Graph()
g.parse("./mappings/NKB_RDF_V3.ttl", format="turtle")

# Extract triples (subject, predicate, object)
triples = []
for s, p, o in g:
    triples.append((str(s), str(p), str(o)))

# Create entity and relation mappings
entities = set([t[0] for t in triples] + [t[2] for t in triples])
relations = set([t[1] for t in triples])

entity_to_idx = {ent: idx for idx, ent in enumerate(entities)}
relation_to_idx = {rel: idx for idx, rel in enumerate(relations)}

# Create edge index and edge attributes for PyTorch Geometric
edge_index = []
edge_type = []

for s, p, o in triples:
    edge_index.append([entity_to_idx[s], entity_to_idx[o]])
    edge_type.append(relation_to_idx[p])

edge_index = torch.tensor(edge_index).t().contiguous()
edge_type = torch.tensor(edge_type)

/Users/pranavsingh/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [21]:
len(edge_index[0])

1369675

NameError: name 'Data' is not defined

: 

Loading RDF file...
Extracting triples...
Found 870012 valid triples
Created graph with 268971 entities and 51 relation types
Training model...
Epoch: 000, Loss: 12.2147, Val AUC: 0.5678
Epoch: 001, Loss: 9.4187, Val AUC: 0.5781
Epoch: 002, Loss: 7.3428, Val AUC: 0.5943
Epoch: 003, Loss: 5.7865, Val AUC: 0.6113
Epoch: 004, Loss: 4.5858, Val AUC: 0.6334
Epoch: 005, Loss: 3.6480, Val AUC: 0.6538
Epoch: 006, Loss: 2.9053, Val AUC: 0.6729
Epoch: 007, Loss: 2.3428, Val AUC: 0.6897
Epoch: 008, Loss: 1.9302, Val AUC: 0.7056
Epoch: 009, Loss: 1.6431, Val AUC: 0.7163
Epoch: 010, Loss: 1.4525, Val AUC: 0.7250
Epoch: 011, Loss: 1.3313, Val AUC: 0.7342
Epoch: 012, Loss: 1.2571, Val AUC: 0.7415
Epoch: 013, Loss: 1.2121, Val AUC: 0.7460
Epoch: 014, Loss: 1.1845, Val AUC: 0.7495
Epoch: 015, Loss: 1.1684, Val AUC: 0.7517
Epoch: 016, Loss: 1.1585, Val AUC: 0.7528
Epoch: 017, Loss: 1.1521, Val AUC: 0.7534
Epoch: 018, Loss: 1.1475, Val AUC: 0.7545
Epoch: 019, Loss: 1.1438, Val AUC: 0.7538
Saving model an

In [6]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv
import pandas as pd
import pickle
import os
import time
import heapq
import numpy as np

# Define RGCN model (same as before)
class RGCN(torch.nn.Module):
    def __init__(self, num_entities, num_relations, embedding_dim=64, num_bases=4):
        super(RGCN, self).__init__()
        self.entity_embedding = torch.nn.Embedding(num_entities, embedding_dim)
        self.relation_embedding = torch.nn.Embedding(num_relations, embedding_dim)
        
        # Simpler architecture with fewer parameters
        self.conv1 = RGCNConv(embedding_dim, embedding_dim, num_relations, num_bases=num_bases)
        
    def forward(self, edge_index, edge_type):
        x = self.entity_embedding.weight
        x = self.conv1(x, edge_index, edge_type)
        x = F.relu(x)
        return x
    
    def decode(self, z, edge_index, sigmoid=True):
        # Simple dot product decoder
        value = (z[edge_index[0]] * z[edge_index[1]]).sum(dim=1)
        return torch.sigmoid(value) if sigmoid else value

# Load model and data
def load_model_and_data(model_dir="model_output"):
    # Load edge index and edge type
    edge_index = torch.load(os.path.join(model_dir, "edge_index.pt"))
    edge_type = torch.load(os.path.join(model_dir, "edge_type.pt"))
    
    # Load entity and relation mappings
    with open(os.path.join(model_dir, "idx_to_entity.pkl"), "rb") as f:
        idx_to_entity = pickle.load(f)
    
    with open(os.path.join(model_dir, "idx_to_relation.pkl"), "rb") as f:
        idx_to_relation = pickle.load(f)
    
    # Load metadata
    with open(os.path.join(model_dir, "metadata.pkl"), "rb") as f:
        metadata = pickle.load(f)
    
    num_entities = metadata["num_entities"]
    num_relations = metadata["num_relations"]
    
    # Initialize model
    model = RGCN(num_entities=num_entities, num_relations=num_relations)
    
    # Load model state
    model.load_state_dict(torch.load(os.path.join(model_dir, "model.pt")))
    model.eval()  # Set model to evaluation mode
    
    print(f"Loaded model with {num_entities} entities and {num_relations} relation types")
    
    return (model, edge_index, edge_type, 
            num_entities, num_relations, 
            idx_to_entity, idx_to_relation)

# Ultra-fast link prediction - completely vectorized approach
def predict_links_ultrafast(model, edge_index, edge_type, num_entities, 
                          idx_to_entity, idx_to_relation, top_k=100, 
                          entity_sample_size=500, batch_size=50):
    """
    Ultra-fast link prediction using vectorized operations and entity sampling
    """
    print("Starting ultra-fast link prediction...")
    start_time = time.time()
    
    with torch.no_grad():
        # Generate node embeddings
        z = model(edge_index, edge_type)
        print(f"Generated embeddings in {time.time() - start_time:.2f} seconds")
        
        # Convert existing edges to set for filtering
        existing_edges = set()
        for i in range(edge_index.size(1)):
            src, dst = edge_index[0, i].item(), edge_index[1, i].item()
            existing_edges.add((src, dst))
        
        print(f"Created existing edges set in {time.time() - start_time:.2f} seconds")
        
        # Sample a subset of source entities
        num_to_sample = min(entity_sample_size, num_entities)
        sampled_sources = torch.randperm(num_entities)[:num_to_sample].tolist()
        
        # Prepare a global heap for top predictions
        top_predictions = []
        
        # Process source entities in small batches
        for batch_start in range(0, len(sampled_sources), batch_size):
            batch_end = min(batch_start + batch_size, len(sampled_sources))
            batch_sources = sampled_sources[batch_start:batch_end]
            
            batch_start_time = time.time()
            print(f"Processing batch {batch_start//batch_size + 1}/{(len(sampled_sources) + batch_size - 1)//batch_size}...")
            
            # Process each source in the batch
            for src in batch_sources:
                # Get the embedding for this source entity
                src_emb = z[src].unsqueeze(0)  # Shape: [1, embedding_dim]
                
                # Calculate similarity with all potential target entities in one go
                # This is much faster than doing it one by one
                # Shape: [1, embedding_dim] × [embedding_dim, num_entities] = [1, num_entities]
                similarity_scores = torch.matmul(src_emb, z.t()).squeeze(0)
                
                # Convert to list for faster processing
                scores_list = similarity_scores.tolist()
                
                # Find potential targets (excluding existing edges)
                for dst in range(num_entities):
                    if dst == src or (src, dst) in existing_edges:
                        continue
                        
                    score = scores_list[dst]
                    
                    # Use a heap to efficiently maintain top-k predictions
                    if len(top_predictions) < top_k:
                        heapq.heappush(top_predictions, (score, src, dst))
                    elif score > top_predictions[0][0]:
                        heapq.heappushpop(top_predictions, (score, src, dst))
            
            print(f"Batch processed in {time.time() - batch_start_time:.2f} seconds")
        
        # Sort the heap by score (descending)
        sorted_predictions = sorted(top_predictions, reverse=True)
        
        # Format the final predictions
        predictions = []
        for score, src, dst in sorted_predictions:
            predictions.append({
                'source': idx_to_entity[src],
                'target': idx_to_entity[dst],
                'score': score
            })
        
        end_time = time.time()
        print(f"Total prediction time: {end_time - start_time:.2f} seconds")
        
        return predictions

# Alternative approach - sample both entity AND relation types
def predict_links_with_relation_types(model, edge_index, edge_type, num_entities, num_relations,
                                    idx_to_entity, idx_to_relation, top_k=100,
                                    entity_sample_size=300, relation_sample_size=5):
    """
    Link prediction that explicitly considers relation types
    """
    print("Starting relation-aware link prediction...")
    start_time = time.time()
    
    with torch.no_grad():
        # Generate node embeddings
        z = model(edge_index, edge_type)
        relation_embeddings = model.relation_embedding.weight
        print(f"Generated embeddings in {time.time() - start_time:.2f} seconds")
        
        # Convert existing edges to set for filtering
        existing_edges = set()
        for i in range(edge_index.size(1)):
            src, dst = edge_index[0, i].item(), edge_index[1, i].item()
            rel = edge_type[i].item()
            existing_edges.add((src, rel, dst))
        
        print(f"Created existing edges set in {time.time() - start_time:.2f} seconds")
        
        # Sample entities and relations
        entity_sample = torch.randperm(num_entities)[:entity_sample_size].tolist()
        relation_sample = torch.randperm(num_relations)[:relation_sample_size].tolist()
        
        print(f"Sampled {len(entity_sample)} entities and {len(relation_sample)} relation types")
        
        # Use a heap for top predictions
        top_predictions = []
        
        # Process combinations
        for src in entity_sample:
            src_emb = z[src]
            
            for rel in relation_sample:
                rel_emb = relation_embeddings[rel]
                
                # TransE-style scoring
                # Predicted tail = head + relation
                predicted_tail = src_emb + rel_emb
                
                # Calculate similarity to all entities
                similarities = torch.matmul(predicted_tail.unsqueeze(0), z.t()).squeeze(0)
                
                # Get top potential targets for this (src, rel) pair
                values, indices = torch.topk(similarities, k=min(10, num_entities))
                
                for i in range(len(indices)):
                    dst = indices[i].item()
                    score = values[i].item()
                    
                    # Skip existing edges and self-loops
                    if (src, rel, dst) in existing_edges or src == dst:
                        continue
                    
                    # Update top predictions
                    if len(top_predictions) < top_k:
                        heapq.heappush(top_predictions, (score, src, rel, dst))
                    elif score > top_predictions[0][0]:
                        heapq.heappushpop(top_predictions, (score, src, rel, dst))
        
        # Sort predictions
        sorted_predictions = sorted(top_predictions, reverse=True)
        
        # Format results
        predictions = []
        for score, src, rel, dst in sorted_predictions:
            predictions.append({
                'source': idx_to_entity[src],
                'relation': idx_to_relation[rel],
                'target': idx_to_entity[dst],
                'score': score
            })
        
        end_time = time.time()
        print(f"Total prediction time: {end_time - start_time:.2f} seconds")
        
        return predictions

# Main function that runs the link prediction
def run_link_prediction(model_dir="model_output", method="ultrafast", top_k=100, output_file=None):
    """Run link prediction with the specified method"""
    # Load model and data
    (model, edge_index, edge_type, 
     num_entities, num_relations, 
     idx_to_entity, idx_to_relation) = load_model_and_data(model_dir)
    
    # Choose prediction method
    start_time = time.time()
    
    if method == "ultrafast":
        predictions = predict_links_ultrafast(
            model, edge_index, edge_type, num_entities, 
            idx_to_entity, idx_to_relation, 
            top_k=top_k, entity_sample_size=500, batch_size=50
        )
    elif method == "with_relations":
        predictions = predict_links_with_relation_types(
            model, edge_index, edge_type, num_entities, num_relations,
            idx_to_entity, idx_to_relation, 
            top_k=top_k, entity_sample_size=300, relation_sample_size=5
        )
    else:
        raise ValueError(f"Unknown method: {method}")
    
    end_time = time.time()
    
    # Output results
    if output_file:
        pd.DataFrame(predictions).to_csv(output_file, index=False)
        print(f"Saved predictions to {output_file}")
    
    # Print summary stats
    print(f"\nCompleted link prediction using {method} method")
    print(f"Total time: {end_time - start_time:.2f} seconds")
    print(f"Found {len(predictions)} high-scoring potential links")
    
    # Print top 5 predictions
    print("\nTop 5 predictions:")
    for i, pred in enumerate(predictions[:5]):
        if method == "with_relations":
            print(f"{i+1}. {pred['source']} --[{pred['relation']}]--> {pred['target']} (score: {pred['score']:.4f})")
        else:
            print(f"{i+1}. {pred['source']} --> {pred['target']} (score: {pred['score']:.4f})")
    
    return predictions

if __name__ == "__main__":
    # Choose method: "ultrafast" or "with_relations"
    method = "ultrafast"
    
    # Run prediction
    predictions = run_link_prediction(
        method=method,
        top_k=100,
        output_file=f"predicted_links_{method}.csv"
    )

/var/folders/ps/cxk7gpds70390sk1wh4zjd240000gn/T/ipykernel_51105/500442765.py:35: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  edge_index = torch.load(os.path.join(model_di

Loaded model with 268971 entities and 51 relation types
Starting ultra-fast link prediction...
Generated embeddings in 1.14 seconds
Created existing edges set in 5.04 seconds
Processing batch 1/10...
Batch processed in 2.72 seconds
Processing batch 2/10...
Batch processed in 2.68 seconds
Processing batch 3/10...
Batch processed in 2.67 seconds
Processing batch 4/10...
Batch processed in 2.69 seconds
Processing batch 5/10...
Batch processed in 2.70 seconds
Processing batch 6/10...
Batch processed in 2.72 seconds
Processing batch 7/10...
Batch processed in 2.71 seconds
Processing batch 8/10...
Batch processed in 2.68 seconds
Processing batch 9/10...
Batch processed in 2.71 seconds
Processing batch 10/10...
Batch processed in 2.62 seconds
Total prediction time: 31.95 seconds
Saved predictions to predicted_links_ultrafast.csv

Completed link prediction using ultrafast method
Total time: 32.03 seconds
Found 100 high-scoring potential links

Top 5 predictions:
1. 210945 --> nan (score: 21.52

In [7]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv
import pandas as pd
import pickle
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import re
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import random

# Define RGCN model (same as before)
class RGCN(torch.nn.Module):
    def __init__(self, num_entities, num_relations, embedding_dim=64, num_bases=4):
        super(RGCN, self).__init__()
        self.entity_embedding = torch.nn.Embedding(num_entities, embedding_dim)
        self.relation_embedding = torch.nn.Embedding(num_relations, embedding_dim)
        
        # Simpler architecture with fewer parameters
        self.conv1 = RGCNConv(embedding_dim, embedding_dim, num_relations, num_bases=num_bases)
        
    def forward(self, edge_index, edge_type):
        x = self.entity_embedding.weight
        x = self.conv1(x, edge_index, edge_type)
        x = F.relu(x)
        return x

# Load model and data
def load_model_and_data(model_dir="model_output"):
    # Load edge index and edge type
    edge_index = torch.load(os.path.join(model_dir, "edge_index.pt"))
    edge_type = torch.load(os.path.join(model_dir, "edge_type.pt"))
    
    # Load entity and relation mappings
    with open(os.path.join(model_dir, "idx_to_entity.pkl"), "rb") as f:
        idx_to_entity = pickle.load(f)
    
    with open(os.path.join(model_dir, "idx_to_relation.pkl"), "rb") as f:
        idx_to_relation = pickle.load(f)
    
    # Load metadata
    with open(os.path.join(model_dir, "metadata.pkl"), "rb") as f:
        metadata = pickle.load(f)
    
    num_entities = metadata["num_entities"]
    num_relations = metadata["num_relations"]
    
    # Create reverse mappings
    entity_to_idx = {entity: idx for idx, entity in idx_to_entity.items()}
    relation_to_idx = {relation: idx for idx, relation in idx_to_relation.items()}
    
    # Initialize model
    model = RGCN(num_entities=num_entities, num_relations=num_relations)
    
    # Load model state
    model.load_state_dict(torch.load(os.path.join(model_dir, "model.pt")))
    model.eval()  # Set model to evaluation mode
    
    print(f"Loaded model with {num_entities} entities and {num_relations} relation types")
    
    return (model, edge_index, edge_type, 
            num_entities, num_relations, 
            idx_to_entity, idx_to_relation,
            entity_to_idx, relation_to_idx)

# Debug function to check entity URI consistency
def debug_entity_issues(idx_to_entity):
    """Check for problematic entity URIs"""
    print("\nDebugging entity URIs...")
    
    # Check for NaN or problematic values
    problematic = []
    nan_values = []
    numeric_values = []
    
    for idx, uri in idx_to_entity.items():
        # Check if the URI is a string
        if not isinstance(uri, str):
            problematic.append((idx, uri))
            
            # Check if it's NaN
            if isinstance(uri, float) and np.isnan(uri):
                nan_values.append(idx)
            # Check if it's a numeric value
            elif isinstance(uri, (int, float)):
                numeric_values.append(idx)
    
    # Print summary
    if problematic:
        print(f"Found {len(problematic)} problematic entity URIs:")
        print(f"  - NaN values: {len(nan_values)}")
        print(f"  - Numeric values: {len(numeric_values)}")
        print(f"  - Other non-string values: {len(problematic) - len(nan_values) - len(numeric_values)}")
        
        # Print some examples
        print("\nExample problematic entries:")
        for idx, uri in random.sample(problematic, min(5, len(problematic))):
            print(f"  Index {idx}: {uri} (type: {type(uri)})")
    else:
        print("All entity URIs are strings - no issues detected.")
    
    return {
        'problematic': problematic,
        'nan_values': nan_values,
        'numeric_values': numeric_values
    }

# Determine node types based on URI patterns with better handling
def determine_node_types(idx_to_entity):
    """Determine types of nodes based on URI patterns, handling non-string values"""
    print("\nDetermining node types based on URI patterns...")
    
    # Define patterns for different node types
    patterns = {
        'nanomaterial': [r'nanomaterial', r'material', r'nano', r'particle', r'nanoparticle'],
        'property': [r'property', r'characteristic', r'size', r'shape', r'diameter', r'charge'],
        'experiment': [r'experiment', r'assay', r'test', r'measurement', r'procedure', r'protocol'],
        'effect': [r'effect', r'outcome', r'result', r'impact', r'toxicity', r'response'],
        'biological': [r'cell', r'organism', r'tissue', r'protein', r'gene', r'biological'],
        'environmental': [r'environment', r'water', r'soil', r'air', r'aquatic', r'terrestrial'],
        'numeric': [r'^[\d\.]+$']  # For entities that are just numbers
    }
    
    # Function to get node type with better handling of non-string values
    def get_node_type(uri):
        # Handle non-string URIs
        if not isinstance(uri, str):
            if isinstance(uri, float) and np.isnan(uri):
                return 'nan'
            elif isinstance(uri, (int, float)):
                return 'numeric'
            else:
                return 'unknown'
        
        # Process string URIs
        uri_lower = uri.lower()
        
        # Check for purely numeric strings
        if re.match(r'^[\d\.]+$', uri):
            return 'numeric'
        
        # Check other patterns
        for node_type, pattern_list in patterns.items():
            for pattern in pattern_list:
                if re.search(pattern, uri_lower):
                    return node_type
        
        # Check for URIs with specific formats
        if 'http://' in uri or 'https://' in uri:
            return 'uri'
        
        return 'other'  # Default type
    
    # Determine type for each node
    node_types = {}
    type_counts = {}
    
    for idx, uri in idx_to_entity.items():
        node_type = get_node_type(uri)
        node_types[idx] = node_type
        
        # Count occurrences of each type
        if node_type not in type_counts:
            type_counts[node_type] = 0
        type_counts[node_type] += 1
    
    # Print type statistics
    print("\nNode type distribution:")
    for node_type, count in sorted(type_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"  {node_type}: {count} nodes")
    
    return node_types, type_counts

# Analyze and debug the graph structure
def analyze_graph_structure(edge_index, edge_type, idx_to_entity, node_types):
    """Analyze the graph structure for patterns and issues"""
    print("\nAnalyzing graph structure...")
    
    # Count connections between different node types
    type_connections = {}
    
    for i in range(edge_index.size(1)):
        src = edge_index[0, i].item()
        dst = edge_index[1, i].item()
        rel = edge_type[i].item()
        
        src_type = node_types.get(src, 'unknown')
        dst_type = node_types.get(dst, 'unknown')
        
        key = (src_type, dst_type)
        if key not in type_connections:
            type_connections[key] = 0
        type_connections[key] += 1
    
    # Print top connection patterns
    print("\nTop connection patterns between node types:")
    for (src_type, dst_type), count in sorted(type_connections.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"  {src_type} → {dst_type}: {count} connections")
    
    # Check for isolated nodes (no connections)
    connected_nodes = set(edge_index[0].tolist() + edge_index[1].tolist())
    all_nodes = set(idx_to_entity.keys())
    isolated_nodes = all_nodes - connected_nodes
    
    print(f"\nFound {len(isolated_nodes)} isolated nodes (no connections)")
    
    # Check node degrees (number of connections)
    node_degrees = {}
    for node in all_nodes:
        in_degree = (edge_index[1] == node).sum().item()
        out_degree = (edge_index[0] == node).sum().item()
        node_degrees[node] = (in_degree, out_degree)
    
    # Group by degree
    degree_groups = {
        'no_connections': 0,
        '1-5': 0,
        '6-20': 0,
        '21-100': 0,
        '100+': 0
    }
    
    for node, (in_deg, out_deg) in node_degrees.items():
        total_deg = in_deg + out_deg
        if total_deg == 0:
            degree_groups['no_connections'] += 1
        elif total_deg <= 5:
            degree_groups['1-5'] += 1
        elif total_deg <= 20:
            degree_groups['6-20'] += 1
        elif total_deg <= 100:
            degree_groups['21-100'] += 1
        else:
            degree_groups['100+'] += 1
    
    print("\nNode degree distribution:")
    for group, count in degree_groups.items():
        print(f"  {group}: {count} nodes")
    
    return {
        'type_connections': type_connections,
        'isolated_nodes': isolated_nodes,
        'node_degrees': node_degrees,
        'degree_groups': degree_groups
    }

# Prepare embeddings for visualization and classification
def prepare_embeddings(model, edge_index, edge_type, node_types):
    """Generate embeddings and prepare for visualization and classification"""
    print("\nGenerating node embeddings...")
    
    # Generate node embeddings
    with torch.no_grad():
        embeddings = model(edge_index, edge_type)
    
    # Check for NaN values in embeddings
    nan_indices = torch.isnan(embeddings).any(dim=1).nonzero().squeeze().tolist()
    if isinstance(nan_indices, int):  # Handle case of single NaN index
        nan_indices = [nan_indices]
    
    if nan_indices:
        print(f"Warning: Found {len(nan_indices)} embeddings with NaN values")
        for idx in nan_indices[:5]:  # Print first few
            print(f"  Node {idx} has NaN in embedding")
    
    # Normalize embeddings for better visualization
    embeddings_norm = F.normalize(embeddings, p=2, dim=1)
    
    # Group embeddings by node type
    embeddings_by_type = {}
    for node_idx, node_type in node_types.items():
        if node_type not in embeddings_by_type:
            embeddings_by_type[node_type] = []
        embeddings_by_type[node_type].append((node_idx, embeddings[node_idx]))
    
    # Sample embeddings for visualization (using at most 1000 per type)
    X_samples = []
    y_samples = []
    id_samples = []
    
    for type_idx, (node_type, embed_list) in enumerate(embeddings_by_type.items()):
        # Sample at most 1000 per type
        sample_size = min(1000, len(embed_list))
        samples = random.sample(embed_list, sample_size)
        
        for node_idx, embed in samples:
            X_samples.append(embed)
            y_samples.append(type_idx)
            id_samples.append(node_idx)
    
    X = torch.stack(X_samples)
    y = torch.tensor(y_samples)
    ids = torch.tensor(id_samples)
    
    type_to_idx = {t: i for i, t in enumerate(embeddings_by_type.keys())}
    idx_to_type = {i: t for t, i in type_to_idx.items()}
    
    return {
        'embeddings': embeddings,
        'embeddings_norm': embeddings_norm,
        'X': X,
        'y': y,
        'ids': ids,
        'nan_indices': nan_indices,
        'type_to_idx': type_to_idx,
        'idx_to_type': idx_to_type
    }

# Visualize embeddings with t-SNE
def visualize_embeddings(embedding_data, idx_to_entity, output_file="node_embeddings.png"):
    """Visualize node embeddings using t-SNE"""
    print("\nVisualizing embeddings with t-SNE...")
    
    X = embedding_data['X']
    y = embedding_data['y']
    ids = embedding_data['ids']
    idx_to_type = embedding_data['idx_to_type']
    
    # Apply t-SNE for dimensionality reduction
    tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
    X_tsne = tsne.fit_transform(X.numpy())
    
    # Create scatter plot
    plt.figure(figsize=(12, 10))
    
    # Colors for different classes
    colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 
              'tab:purple', 'tab:brown', 'tab:pink', 'tab:gray', 'tab:olive', 'tab:cyan']
    
    # Plot each class
    for i, label in enumerate(idx_to_type.keys()):
        mask = y == label
        plt.scatter(
            X_tsne[mask, 0], X_tsne[mask, 1], 
            label=idx_to_type[label],
            alpha=0.7,
            color=colors[i % len(colors)]
        )
    
    plt.legend()
    plt.title('t-SNE Visualization of Node Embeddings')
    plt.tight_layout()
    plt.savefig(output_file)
    plt.close()
    
    print(f"Visualization saved to {output_file}")
    
    # Generate interactive HTML version with plotly if available
    try:
        import plotly.express as px
        import plotly.io as pio
        
        # Prepare data
        df = pd.DataFrame({
            'x': X_tsne[:, 0],
            'y': X_tsne[:, 1],
            'type': [idx_to_type[label.item()] for label in y],
            'id': [ids[i].item() for i in range(len(ids))],
            'entity': [str(idx_to_entity.get(ids[i].item(), "Unknown")) for i in range(len(ids))]
        })
        
        # Create interactive plot
        fig = px.scatter(
            df, x='x', y='y', color='type', 
            hover_data=['id', 'entity'],
            title='Interactive t-SNE Visualization of Node Embeddings'
        )
        
        # Save interactive plot
        html_file = output_file.replace('.png', '.html')
        pio.write_html(fig, html_file)
        print(f"Interactive visualization saved to {html_file}")
    except ImportError:
        print("Plotly not available - skipping interactive visualization")

# Train and evaluate a simple classifier
def train_node_classifier(embedding_data, hidden_dim=32, test_size=0.2, random_state=42):
    """Train a simple classifier on node embeddings"""
    print("\nTraining node classifier...")
    
    X = embedding_data['X']
    y = embedding_data['y']
    
    # Skip if insufficient data
    if len(torch.unique(y)) < 2:
        print("Insufficient classes for classification - skipping")
        return None, 0, "N/A"
    
    # Split data into train and test sets
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    
    # Get dimensions
    input_dim = X.shape[1]
    num_classes = len(torch.unique(y))
    
    # Create classifier (simple MLP)
    classifier = torch.nn.Sequential(
        torch.nn.Linear(input_dim, hidden_dim),
        torch.nn.ReLU(),
        torch.nn.Dropout(0.2),
        torch.nn.Linear(hidden_dim, num_classes)
    )
    
    # Define loss and optimizer
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(classifier.parameters(), lr=0.01)
    
    # Training loop
    num_epochs = 100
    for epoch in range(num_epochs):
        classifier.train()
        
        # Forward pass
        outputs = classifier(X_train)
        loss = criterion(outputs, y_train)
        
        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Print progress every 10 epochs
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}")
    
    # Evaluate on test set
    classifier.eval()
    with torch.no_grad():
        outputs = classifier(X_test)
        _, predicted = torch.max(outputs, 1)
        accuracy = accuracy_score(y_test.numpy(), predicted.numpy())
        report = classification_report(y_test.numpy(), predicted.numpy())
        
        print(f"\nTest accuracy: {accuracy:.4f}")
        print("\nClassification report:")
        print(report)
    
    return classifier, accuracy, report

# Test prediction with sample nodes
def test_prediction_with_samples(model, edge_index, edge_type, idx_to_entity, node_types):
    """Test predictions with sample nodes to debug issues"""
    print("\nTesting predictions with sample nodes...")
    
    # Generate embeddings
    with torch.no_grad():
        z = model(edge_index, edge_type)
    
    # Get existing edges set
    existing_edges = set()
    for i in range(edge_index.size(1)):
        src, dst = edge_index[0, i].item(), edge_index[1, i].item()
        existing_edges.add((src, dst))
    
    # Sample a few nodes from different types to test predictions
    samples = []
    for node_type in ['nanomaterial', 'property', 'experiment', 'biological', 'other']:
        type_nodes = [idx for idx, t in node_types.items() if t == node_type]
        if type_nodes:
            samples.extend(random.sample(type_nodes, min(2, len(type_nodes))))
    
    if not samples:
        samples = random.sample(list(idx_to_entity.keys()), min(10, len(idx_to_entity)))
    
    # Test predictions for each sample
    results = []
    
    for src in samples:
        # Get source embedding and entity
        src_emb = z[src]
        src_entity = idx_to_entity.get(src, "Unknown")
        src_type = node_types.get(src, "Unknown")
        
        # Find potential targets (excluding existing connections)
        potential_targets = [dst for dst in range(len(z)) if dst != src and (src, dst) not in existing_edges]
        
        # Sample a subset of potential targets
        sample_targets = random.sample(potential_targets, min(100, len(potential_targets)))
        
        # Create potential edges tensor
        potential_edges = torch.tensor([[src, dst] for dst in sample_targets], dtype=torch.long).t()
        
        # Compute scores
        with torch.no_grad():
            scores = (z[potential_edges[0]] * z[potential_edges[1]]).sum(dim=1)
        
        # Get top 5 predictions
        if len(scores) > 0:
            top_values, top_indices = torch.topk(scores, k=min(5, len(scores)))
            
            # Format predictions
            predictions = []
            for i in range(len(top_indices)):
                dst_idx = sample_targets[top_indices[i].item()]
                dst_entity = idx_to_entity.get(dst_idx, "Unknown")
                dst_type = node_types.get(dst_idx, "Unknown")
                
                predictions.append({
                    'dst_idx': dst_idx,
                    'dst_entity': dst_entity,
                    'dst_type': dst_type,
                    'score': top_values[i].item()
                })
            
            results.append({
                'src_idx': src,
                'src_entity': src_entity,
                'src_type': src_type,
                'predictions': predictions
            })
    
    # Print prediction results
    print("\nSample prediction results:")
    for result in results:
        print(f"\nSource: {result['src_entity']} (Type: {result['src_type']})")
        print("Top predictions:")
        for i, pred in enumerate(result['predictions']):
            print(f"  {i+1}. {pred['dst_entity']} (Type: {pred['dst_type']}, Score: {pred['score']:.4f})")
    
    return results

# Main execution function
def run_analysis(model_dir="model_output"):
    """Main function to run analysis on trained model"""
    # Load model and data
    (model, edge_index, edge_type, 
     num_entities, num_relations, 
     idx_to_entity, idx_to_relation,
     entity_to_idx, relation_to_idx) = load_model_and_data(model_dir)
    
    # Debug entity issues
    entity_issues = debug_entity_issues(idx_to_entity)
    
    # Determine node types
    node_types, type_counts = determine_node_types(idx_to_entity)
    
    # Analyze graph structure
    graph_analysis = analyze_graph_structure(edge_index, edge_type, idx_to_entity, node_types)
    
    # Prepare embeddings
    embedding_data = prepare_embeddings(model, edge_index, edge_type, node_types)
    
    # Visualize embeddings
    visualize_embeddings(embedding_data, idx_to_entity)
    
    # Train classifier
    classifier, accuracy, report = train_node_classifier(embedding_data)
    
    # Test predictions
    prediction_tests = test_prediction_with_samples(model, edge_index, edge_type, idx_to_entity, node_types)
    
    # Return results
    return {
        'entity_issues': entity_issues,
        'node_types': node_types,
        'type_counts': type_counts,
        'graph_analysis': graph_analysis,
        'embedding_data': embedding_data,
        'classifier_accuracy': accuracy,
        'classification_report': report,
        'prediction_tests': prediction_tests
    }

if __name__ == "__main__":
    # Run analysis
    results = run_analysis()
    
    # Print summary
    print("\nAnalysis complete!")

/var/folders/ps/cxk7gpds70390sk1wh4zjd240000gn/T/ipykernel_51105/3845120343.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  edge_index = torch.load(os.path.join(model_d

Loaded model with 268971 entities and 51 relation types

Debugging entity URIs...
All entity URIs are strings - no issues detected.

Determining node types based on URI patterns...

Node type distribution:
  numeric: 130511 nodes
  uri: 85142 nodes
  effect: 24720 nodes
  experiment: 22399 nodes
  other: 4421 nodes
  nanomaterial: 1615 nodes
  biological: 61 nodes
  property: 52 nodes
  environmental: 50 nodes

Analyzing graph structure...

Top connection patterns between node types:
  uri → other: 186241 connections
  uri → uri: 159341 connections
  effect → other: 148332 connections
  uri → numeric: 145847 connections
  effect → numeric: 52133 connections
  experiment → other: 43854 connections
  experiment → uri: 41548 connections
  effect → experiment: 24755 connections
  experiment → numeric: 22329 connections
  effect → uri: 15523 connections

Found 0 isolated nodes (no connections)

Node degree distribution:
  no_connections: 0 nodes
  1-5: 157375 nodes
  6-20: 110108 nodes
  21

/Users/pranavsingh/anaconda3/lib/python3.11/site-packages/plotly/express/_core.py:1979: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  sf: grouped.get_group(s if len(s) > 1 else s[0])


Interactive visualization saved to node_embeddings.html

Training node classifier...
Epoch 10/100, Loss: 1.9855
Epoch 20/100, Loss: 1.9126
Epoch 30/100, Loss: 1.9022
Epoch 40/100, Loss: 1.8788
Epoch 50/100, Loss: 1.8690
Epoch 60/100, Loss: 1.8583
Epoch 70/100, Loss: 1.8535
Epoch 80/100, Loss: 1.8485
Epoch 90/100, Loss: 1.8408
Epoch 100/100, Loss: 1.8354

Test accuracy: 0.1849

Classification report:
              precision    recall  f1-score   support

           0       0.22      0.21      0.22       200
           1       0.17      0.28      0.21       200
           2       0.24      0.19      0.21       200
           3       0.21      0.12      0.15       200
           4       0.16      0.28      0.20       200
           5       0.00      0.00      0.00        11
           6       0.00      0.00      0.00        10
           7       0.00      0.00      0.00        12
           8       0.14      0.06      0.08       200

    accuracy                           0.18      1233
 

/Users/pranavsingh/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning:

Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/Users/pranavsingh/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning:

Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/Users/pranavsingh/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning:

Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.




Sample prediction results:

Source: http://example.org/material/16 (Type: nanomaterial)
Top predictions:
  1. 1340886 (Type: numeric, Score: 0.1687)
  2. 1275626 (Type: numeric, Score: 0.1553)
  3. 195424 (Type: numeric, Score: 0.1512)
  4. 1284223 (Type: numeric, Score: 0.1413)
  5. http://example.org/parameters/81578 (Type: uri, Score: 0.1351)

Source: this nano scaled material is composed of a graphene oxide core with no information on the shell and no information on the coating. no value with no units was reported for the particle size (diameter). this material was obtained from cheap tubes, with product/lot number unreported, designation 4. (Type: nanomaterial)
Top predictions:
  1. 1310507 (Type: numeric, Score: 0.2562)
  2. http://example.org/parameters/41587 (Type: uri, Score: 0.1708)
  3. http://example.org/result/23225 (Type: effect, Score: 0.1248)
  4. 1321247 (Type: numeric, Score: 0.1224)
  5. http://example.org/result/4085 (Type: effect, Score: 0.1047)

Source: hydrodyna

In [8]:
import rdflib
from rdflib import Graph, URIRef
from collections import defaultdict, Counter
import pandas as pd

def explore_rdf_structure(rdf_file_path, format="turtle"):
    """
    Load and explore the structure of an RDF file.
    
    Parameters:
    rdf_file_path (str): Path to the RDF file
    format (str): Format of the RDF file (e.g., "turtle", "xml", "n3")
    """
    print(f"Loading RDF from {rdf_file_path}...")
    g = Graph()
    g.parse(rdf_file_path, format=format)
    
    print(f"Loaded {len(g)} triples")
    
    # 1. Basic Statistics
    print("\n==== Basic Graph Statistics ====")
    
    # Count unique subjects, predicates, objects
    subjects = set(g.subjects())
    predicates = set(g.predicates())
    objects = set(g.objects())
    
    print(f"Unique subjects: {len(subjects)}")
    print(f"Unique predicates: {len(predicates)}")
    print(f"Unique objects: {len(objects)}")
    
    # Count entity types
    subject_types = Counter([type(s).__name__ for s in subjects])
    object_types = Counter([type(o).__name__ for o in objects])
    
    print("\nSubject types:")
    for t, count in subject_types.items():
        print(f"  {t}: {count}")
    
    print("\nObject types:")
    for t, count in object_types.items():
        print(f"  {t}: {count}")
    
    # 2. Examine Predicates (Relations)
    print("\n==== Predicates/Relations ====")
    predicate_counts = Counter()
    
    for p in predicates:
        count = len(list(g.triples((None, p, None))))
        predicate_counts[str(p)] = count
    
    # Display top 10 most common predicates
    print("Top 10 most common predicates:")
    for p, count in predicate_counts.most_common(10):
        print(f"  {p} ({count} occurrences)")
    
    # 3. Explore Node Connectivity
    print("\n==== Node Connectivity ====")
    
    # Create dictionaries to store outgoing and incoming connections
    outgoing_edges = defaultdict(list)
    incoming_edges = defaultdict(list)
    
    for s, p, o in g:
        if isinstance(s, URIRef) and (isinstance(o, URIRef) or isinstance(o, rdflib.term.Literal)):
            outgoing_edges[str(s)].append((str(p), str(o)))
        
        if isinstance(o, URIRef) and isinstance(s, URIRef):
            incoming_edges[str(o)].append((str(p), str(s)))
    
    # Find nodes with most outgoing connections
    outgoing_counts = {node: len(edges) for node, edges in outgoing_edges.items()}
    top_outgoing = sorted(outgoing_counts.items(), key=lambda x: x[1], reverse=True)[:5]
    
    print("Nodes with most outgoing connections:")
    for node, count in top_outgoing:
        print(f"  {node}: {count} outgoing connections")
    
    # Find nodes with most incoming connections
    incoming_counts = {node: len(edges) for node, edges in incoming_edges.items()}
    top_incoming = sorted(incoming_counts.items(), key=lambda x: x[1], reverse=True)[:5]
    
    print("\nNodes with most incoming connections:")
    for node, count in top_incoming:
        print(f"  {node}: {count} incoming connections")
    
    # 4. Examine a specific node in detail
    if top_outgoing:
        sample_node = top_outgoing[0][0]
        print(f"\n==== Detailed View of Node: {sample_node} ====")
        
        print("Outgoing connections:")
        predicate_objects = {}
        for p, o in outgoing_edges[sample_node]:
            if p not in predicate_objects:
                predicate_objects[p] = []
            predicate_objects[p].append(o)
        
        for p, objects in predicate_objects.items():
            print(f"  {p}: {len(objects)} values")
            # Show first 3 values for each predicate
            for o in objects[:3]:
                print(f"    -> {o}")
            if len(objects) > 3:
                print(f"    ... and {len(objects) - 3} more")
        
        print("\nIncoming connections:")
        predicate_subjects = {}
        for p, s in incoming_edges.get(sample_node, []):
            if p not in predicate_subjects:
                predicate_subjects[p] = []
            predicate_subjects[p].append(s)
        
        for p, subjects in predicate_subjects.items():
            print(f"  {p}: {len(subjects)} values")
            # Show first 3 values for each predicate
            for s in subjects[:3]:
                print(f"    <- {s}")
            if len(subjects) > 3:
                print(f"    ... and {len(subjects) - 3} more")
    
    # 5. Find isolated nodes (nodes with no connections)
    isolated_subjects = [s for s in subjects if str(s) not in outgoing_edges and str(s) not in incoming_edges]
    isolated_objects = [o for o in objects if isinstance(o, URIRef) and str(o) not in outgoing_edges and str(o) not in incoming_edges]
    
    isolated_nodes = set([str(node) for node in isolated_subjects + isolated_objects])
    print(f"\n==== Isolated Nodes ====")
    print(f"Found {len(isolated_nodes)} isolated nodes")
    if isolated_nodes:
        print("Examples:")
        for node in list(isolated_nodes)[:5]:
            print(f"  {node}")
    
    # 6. Return useful data structures for further analysis
    return {
        "graph": g,
        "subjects": subjects,
        "predicates": predicates,
        "objects": objects,
        "outgoing_edges": outgoing_edges,
        "incoming_edges": incoming_edges,
        "predicate_counts": predicate_counts
    }

# Example: Custom analysis functions using the returned data
def find_path_between_nodes(outgoing_edges, start_node, end_node, max_depth=3):
    """Find a path between two nodes using BFS"""
    queue = [(start_node, [])]
    visited = set()
    
    while queue:
        node, path = queue.pop(0)
        
        if node == end_node:
            return path
        
        if node in visited or len(path) >= max_depth:
            continue
        
        visited.add(node)
        
        for predicate, target in outgoing_edges.get(node, []):
            queue.append((target, path + [(node, predicate, target)]))
    
    return None

def find_similar_nodes(graph_data, target_node, top_n=5):
    """Find nodes with similar connection patterns to the target node"""
    outgoing = graph_data["outgoing_edges"]
    
    # Get predicates used by target node
    target_predicates = set(p for p, _ in outgoing.get(target_node, []))
    
    # Calculate similarity scores
    similarity_scores = {}
    for node in outgoing:
        if node == target_node:
            continue
        
        node_predicates = set(p for p, _ in outgoing.get(node, []))
        # Jaccard similarity of predicates
        intersection = len(target_predicates & node_predicates)
        union = len(target_predicates | node_predicates)
        
        if union > 0:
            similarity_scores[node] = intersection / union
    
    # Return top N similar nodes
    return sorted(similarity_scores.items(), key=lambda x: x[1], reverse=True)[:top_n]

# Example usage
if __name__ == "__main__":
    rdf_file_path = "./mappings/NKB_RDF_V3.ttl"  # Replace with your RDF file path
    graph_data = explore_rdf_structure(rdf_file_path, format="turtle")
    
    # Further analysis examples:
    if graph_data["outgoing_edges"]:
        # Get two sample nodes
        sample_nodes = list(graph_data["outgoing_edges"].keys())[:2]
        if len(sample_nodes) >= 2:
            start_node, end_node = sample_nodes
            
            print("\n==== Path Finding Example ====")
            print(f"Finding path from {start_node} to {end_node}:")
            path = find_path_between_nodes(graph_data["outgoing_edges"], start_node, end_node)
            if path:
                for s, p, o in path:
                    print(f"  {s} --[{p}]--> {o}")
            else:
                print("  No path found within max depth")
            
            print("\n==== Similar Nodes Example ====")
            print(f"Nodes similar to {start_node}:")
            similar_nodes = find_similar_nodes(graph_data, start_node)
            for node, score in similar_nodes:
                print(f"  {node} (similarity: {score:.2f})")

Loading RDF from ./mappings/NKB_RDF_V3.ttl...
Loaded 1369675 triples

==== Basic Graph Statistics ====
Unique subjects: 290789
Unique predicates: 71
Unique objects: 301399

Subject types:
  BNode: 158579
  URIRef: 132210

Object types:
  BNode: 158579
  Literal: 141696
  URIRef: 1124

==== Predicates/Relations ====
Top 10 most common predicates:
  http://purl.org/dc/terms/identifier (284602 occurrences)
  http://purl.obolibrary.org/obo/OBI_0002110 (176420 occurrences)
  http://www.w3.org/1999/02/22-rdf-syntax-ns#type (132136 occurrences)
  http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C42614 (130957 occurrences)
  http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C68553 (112389 occurrences)
  http://www.w3.org/1999/02/22-rdf-syntax-ns#value (111267 occurrences)
  http://purl.obolibrary.org/obo/OBI_0000070 (107936 occurrences)
  http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C25704 (83233 occurrences)
  http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C54191 (28807 occurrenc

In [9]:
def create_attributed_graph_from_rdf(rdf_file_path):
    g = Graph()
    g.parse(rdf_file_path, format="turtle")
    
    # Group triples by subject to create attributed nodes
    entities = {}
    relationships = []
    
    # First pass: collect attributes for each entity
    for s, p, o in g:
        s_str = str(s)
        p_str = str(p)
        o_str = str(o)
        
        # Skip blank nodes
        if isinstance(s, rdflib.BNode):
            continue
            
        # Determine if this predicate represents an attribute or a relationship
        if (isinstance(o, rdflib.Literal) or 
            p_str.endswith('identifier') or 
            p_str.endswith('name') or
            p_str.endswith('value')):
            # This is an attribute
            if s_str not in entities:
                entities[s_str] = {'uri': s_str, 'attributes': {}}
            entities[s_str]['attributes'][p_str] = o_str
        elif isinstance(o, rdflib.URIRef):
            # This is a relationship between entities
            relationships.append((s_str, p_str, o_str))
    
    # Extract node types
    for s_str, entity in entities.items():
        for p, o in g.predicate_objects(s):
            if str(p) == "http://www.w3.org/1999/02/22-rdf-syntax-ns#type":
                entity['type'] = str(o)
                break
    
    # Create feature vectors for each entity
    entity_features = {}
    all_attribute_keys = set()
    
    # Collect all possible attribute keys
    for entity in entities.values():
        all_attribute_keys.update(entity['attributes'].keys())
    
    # Sort keys for consistent ordering
    all_attribute_keys = sorted(list(all_attribute_keys))
    
    # Create numeric feature vectors
    for uri, entity in entities.items():
        # One-hot encode the entity type
        entity_type = entity.get('type', 'unknown')
        
        # Convert string attributes to numeric features (e.g. using embeddings or one-hot encoding)
        feature_vector = []
        for key in all_attribute_keys:
            if key in entity['attributes']:
                value = entity['attributes'][key]
                # Convert the value to a numeric feature
                # This is a simplified version - you would need more sophisticated conversion
                if isinstance(value, str) and value.replace('.', '', 1).isdigit():
                    feature_vector.append(float(value))
                else:
                    feature_vector.append(1.0)  # Indicator that the attribute exists
            else:
                feature_vector.append(0.0)  # Missing attribute
                
        entity_features[uri] = torch.tensor(feature_vector, dtype=torch.float)
    
    # Create edge index and edge attributes for actual relationships
    entity_to_idx = {uri: i for i, uri in enumerate(entities.keys())}
    edge_index = []
    edge_type = []
    relation_to_idx = {}
    
    for s, p, o in relationships:
        if s in entity_to_idx and o in entity_to_idx:
            if p not in relation_to_idx:
                relation_to_idx[p] = len(relation_to_idx)
                
            edge_index.append([entity_to_idx[s], entity_to_idx[o]])
            edge_type.append(relation_to_idx[p])
    
    # Convert to tensors
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_type = torch.tensor(edge_type, dtype=torch.long)
    
    # Stack all feature vectors in the same order as entity_to_idx
    node_features = torch.stack([
        entity_features[uri] for uri in entity_to_idx.keys()
    ])
    
    return edge_index, edge_type, node_features, entity_to_idx, relation_to_idx

In [25]:
import torch
import rdflib
from rdflib import Graph, URIRef, Namespace
import numpy as np
from collections import defaultdict, Counter
from torch_geometric.data import HeteroData

class HeterogeneousNKBProcessor:
    def __init__(self, rdf_file_path, format="turtle"):
        """Initialize the processor with the RDF data"""
        self.rdf_file_path = rdf_file_path
        self.format = format
        self.setup_namespaces()
        
    def setup_namespaces(self):
        """Set up the namespaces used in the NKB dataset"""
        self.namespaces = {
            'ncit': Namespace('http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#'),
            'obo': Namespace('http://purl.obolibrary.org/obo/'),
            'dcterms': Namespace('http://purl.org/dc/terms/'),
            'npo': Namespace('http://purl.bioontology.org/ontology/npo#'),
            'dc': Namespace('http://purl.org/dc/elements/1.1/'),
            'enm': Namespace('http://purl.enanomapper.org/onto/'),
            'sio': Namespace('http://semanticscience.org/resource/'),
            'edam': Namespace('http://edamontology.org/'),
            'birnlex': Namespace('http://bioontology.org/projects/ontologies/birnlex#'),
            'odo': Namespace('http://purl.dataone.org/odo/'),
            'oboInOwl': Namespace('http://www.geneontology.org/formats/oboInOwl#'),
            'rdf': Namespace('http://www.w3.org/1999/02/22-rdf-syntax-ns#'),
            'rdfs': Namespace('http://www.w3.org/2000/01/rdf-schema#'),
            'bao': Namespace('http://www.bioassayontology.org/bao#')
        }
        
        # Create prefix map for abbreviating URIs
        self.prefix_map = {}
        for prefix, namespace in self.namespaces.items():
            self.prefix_map[str(namespace)] = prefix
    
    def abbreviate_uri(self, uri):
        """Abbreviate URI using namespace prefixes"""
        for namespace, prefix in self.prefix_map.items():
            if uri.startswith(namespace):
                return f"{prefix}:{uri[len(namespace):]}"
        return uri
        
    def process_rdf(self):
        """Process the RDF data into a heterogeneous graph"""
        print(f"Loading RDF file: {self.rdf_file_path}")
        g = Graph()
        
        # Bind namespaces
        for prefix, namespace in self.namespaces.items():
            g.bind(prefix, namespace)
            
        # Parse RDF file
        g.parse(self.rdf_file_path, format=self.format)
        print(f"Loaded {len(g)} triples")
        
        # Print a sample of triples to help debug
        print("\nSample triples (10):")
        sample_count = 0
        for s, p, o in g:
            print(f"Subject: {self.abbreviate_uri(str(s))}")
            print(f"Predicate: {self.abbreviate_uri(str(p))}")
            if isinstance(o, rdflib.Literal):
                print(f"Object: {str(o)[:50]}...")
            else:
                print(f"Object: {self.abbreviate_uri(str(o))}")
            print("---")
            sample_count += 1
            if sample_count >= 10:
                break
        
        # Define node types based on RDF type predicates
        self.node_types = {
            'nanomaterial': str(self.namespaces['npo']['NPO_199']),
            'assay': str(self.namespaces['obo']['OBI_0000070']),
            'result': str(self.namespaces['bao']['BAO_0000179']),
            'parameter': str(self.namespaces['npo']['NPO_1680']),
            'publication': str(self.namespaces['ncit']['C48471']),
            'medium': str(self.namespaces['npo']['NPO_1853']),
            'property': str(self.namespaces['npo']['NPO_1680']),
            'additive': str(self.namespaces['ncit']['C63495']),
            'contaminant': str(self.namespaces['ncit']['C84280']),
            'functional_group': str(self.namespaces['npo']['NPO_174'])
        }
        
        # Define nodebag types
        self.nodebag_types = {
            'synthesis': str(self.namespaces['ncit']['C61408']),
            'outer_diameter': str(self.namespaces['ncit']['C124136']),
            'inner_diameter': str(self.namespaces['ncit']['C101685']),
            'length': str(self.namespaces['ncit']['C25334']),
            'thickness': str(self.namespaces['ncit']['C41145']),
            'surface_area': str(self.namespaces['sio']['CHEMINF_000247']),
            'size_distribution': str(self.namespaces['enm']['ENM_8000292']),
            'hydrodynamic_diameter': str(self.namespaces['npo']['NPO_1915']),
            'surface_charge': str(self.namespaces['npo']['NPO_1812']),
            'purity': str(self.namespaces['npo']['NPO_1345'])
        }
        
        # Identify all entities and their types
        print("Identifying entities and their types...")
        self.entities = {}
        type_pred = self.namespaces['rdf']['type']
        
        # Count entities found by type
        entity_counts = defaultdict(int)
        
        # First, collect all entities with rdf:type predicates
        for s, p, o in g.triples((None, type_pred, None)):
            if isinstance(s, URIRef):
                entity_uri = str(s)
                type_uri = str(o)
                
                # Determine entity type based on the rdf:type value
                entity_type = None
                for typename, typeuri in self.node_types.items():
                    if type_uri == typeuri:
                        entity_type = typename
                        entity_counts[typename] += 1
                        break
                        
                # If it's a nodebag type, determine the specific nodebag type
                if entity_type is None:
                    for bagname, baguri in self.nodebag_types.items():
                        if type_uri == baguri:
                            entity_type = bagname
                            entity_counts[bagname] += 1
                            break
                
                # If we still don't have a type, use a generic "other" type
                if entity_type is None:
                    entity_type = "other"
                    entity_counts["other"] += 1
                
                # Create the entity entry
                if entity_uri not in self.entities:
                    self.entities[entity_uri] = {
                        "uri": entity_uri,
                        "abbreviated_uri": self.abbreviate_uri(entity_uri),
                        "type": entity_type,
                        "attributes": {},
                        "relationships": []
                    }
                else:
                    self.entities[entity_uri]["type"] = entity_type
        
        # Print entity counts by type
        print("\nEntities found by type:")
        for entity_type, count in entity_counts.items():
            print(f"  {entity_type}: {count}")
        
        # Look for relationships even if no rdf:type is present
        # This will catch entities that might not have an explicit type
        print("\nLooking for additional entities without explicit types...")
        additional_count = 0
        
        for s, p, o in g:
            s_str = str(s)
            o_str = str(o)
            
            # Check if subject is not yet in entities but appears as a URI
            if isinstance(s, URIRef) and s_str not in self.entities:
                # Add it as a generic entity
                self.entities[s_str] = {
                    "uri": s_str,
                    "abbreviated_uri": self.abbreviate_uri(s_str),
                    "type": "unknown",
                    "attributes": {},
                    "relationships": []
                }
                additional_count += 1
                
            # Check if object is a URI and not yet in entities
            if isinstance(o, URIRef) and o_str not in self.entities:
                # Add it as a generic entity
                self.entities[o_str] = {
                    "uri": o_str,
                    "abbreviated_uri": self.abbreviate_uri(o_str),
                    "type": "unknown",
                    "attributes": {},
                    "relationships": []
                }
                additional_count += 1
        
        print(f"Added {additional_count} additional entities without explicit types")
        
        # Now collect attributes and relationships for all entities
        print("Collecting attributes and relationships...")
        for s, p, o in g:
            if isinstance(s, URIRef):
                s_str = str(s)
                p_str = str(p)
                
                # Skip if this subject isn't an entity we're tracking
                if s_str not in self.entities:
                    continue
                
                # If the object is a literal, it's an attribute
                if isinstance(o, rdflib.Literal):
                    self.entities[s_str]["attributes"][p_str] = str(o)
                
                # If the object is a URI and it's a tracked entity, it's a relationship
                elif isinstance(o, URIRef) and str(o) in self.entities:
                    self.entities[s_str]["relationships"].append({
                        "predicate": p_str,
                        "abbreviated_predicate": self.abbreviate_uri(p_str),
                        "target": str(o)
                    })
        
        # Count relationships found
        relationship_count = sum(len(e["relationships"]) for e in self.entities.values())
        print(f"Found {relationship_count} relationships between entities")

        # Build dictionary of nodebags connected to main entities
        print("Processing nodebags...")
        self.nodebag_connections = defaultdict(list)
        
        # Find relationships that connect main entities to nodebags
        for entity_uri, entity_data in self.entities.items():
            if entity_data["type"] in self.nodebag_types.keys():
                # This is a nodebag, find what it's connected to
                connected = False
                for s, p, o in g.triples((None, None, URIRef(entity_uri))):
                    if isinstance(s, URIRef) and str(s) in self.entities:
                        parent_uri = str(s)
                        if self.entities[parent_uri]["type"] in self.node_types.keys():
                            # This nodebag is connected to a main entity
                            self.nodebag_connections[parent_uri].append({
                                "nodebag_type": entity_data["type"],
                                "nodebag_uri": entity_uri
                            })
                            connected = True
                
                if not connected:
                    # Handle orphaned nodebags - find if they're referenced in other triples
                    for s, p, o in g.triples((URIRef(entity_uri), None, None)):
                        if isinstance(o, URIRef) and str(o) in self.entities:
                            target_uri = str(o)
                            if self.entities[target_uri]["type"] in self.node_types.keys():
                                # Connect the nodebag to this entity
                                self.nodebag_connections[target_uri].append({
                                    "nodebag_type": entity_data["type"],
                                    "nodebag_uri": entity_uri
                                })
                                connected = True
                                break
        
        # Count entity types after processing
        entity_type_counts = Counter([e["type"] for e in self.entities.values()])
        print("\nEntity type distribution:")
        for type_name, count in entity_type_counts.most_common():
            print(f"  {type_name}: {count}")
        
        # Make sure we have at least some relationships
        if relationship_count == 0:
            print("\nWARNING: No relationships were found between entities.")
            print("This may indicate an issue with the RDF data or how we're parsing it.")
            print("Adding manual relationships for nodebags...")
            
            # Add explicit relationships for nodebags
            added_relationships = 0
            for parent_uri, connections in self.nodebag_connections.items():
                for connection in connections:
                    nodebag_uri = connection["nodebag_uri"]
                    nodebag_type = connection["nodebag_type"]
                    
                    # Add a has_property relationship from parent to nodebag
                    self.entities[parent_uri]["relationships"].append({
                        "predicate": "has_property",
                        "abbreviated_predicate": "has_property",
                        "target": nodebag_uri
                    })
                    added_relationships += 1
            
            print(f"Added {added_relationships} manual relationships for nodebags")
            
            # Check for material - assay relationships
            material_assay_count = 0
            material_uris = [uri for uri, data in self.entities.items() if data["type"] == "nanomaterial"]
            assay_uris = [uri for uri, data in self.entities.items() if data["type"] == "assay"]
            
            # Look for material-assay connections via shared attributes
            material_doi_map = {}
            assay_doi_map = {}
            
            # Find DOIs for materials and assays
            for uri in material_uris:
                for attr, value in self.entities[uri]["attributes"].items():
                    if "doi" in attr.lower() or "publication" in attr.lower():
                        material_doi_map[value] = uri
            
            for uri in assay_uris:
                for attr, value in self.entities[uri]["attributes"].items():
                    if "doi" in attr.lower() or "publication" in attr.lower():
                        if value in material_doi_map:
                            # Found a shared DOI - create relationship
                            material_uri = material_doi_map[value]
                            self.entities[material_uri]["relationships"].append({
                                "predicate": "tested_in",
                                "abbreviated_predicate": "tested_in",
                                "target": uri
                            })
                            material_assay_count += 1
            
            print(f"Added {material_assay_count} material-assay relationships based on shared DOIs")
        
        # Create a list of entity types (for heterogeneous graph)
        self.entity_types = list(set([e["type"] for e in self.entities.values()]))
        
        # Map entities to indices by type
        self.entity_to_idx = {}
        self.idx_to_entity = {}
        self.nodes_by_type = defaultdict(list)
        
        for entity_type in self.entity_types:
            self.entity_to_idx[entity_type] = {}
            self.idx_to_entity[entity_type] = {}
            
            entities_of_type = [uri for uri, data in self.entities.items() 
                            if data["type"] == entity_type]
            
            for idx, uri in enumerate(entities_of_type):
                self.entity_to_idx[entity_type][uri] = idx
                self.idx_to_entity[entity_type][idx] = uri
                self.nodes_by_type[entity_type].append(uri)
        
        # Create feature matrices for each node type
        print("Creating feature matrices...")
        self.feature_matrices = {}
        self.attribute_maps = {}
        
        for entity_type in self.entity_types:
            # Collect all attribute keys for this type
            attribute_keys = set()
            for uri in self.nodes_by_type[entity_type]:
                attribute_keys.update(self.entities[uri]["attributes"].keys())
            
            # Map attribute keys to indices
            self.attribute_maps[entity_type] = {attr: idx for idx, attr in enumerate(sorted(attribute_keys))}
            num_attributes = len(self.attribute_maps[entity_type])
            
            # Create feature matrix
            num_entities = len(self.nodes_by_type[entity_type])
            feature_matrix = np.zeros((num_entities, max(1, num_attributes)), dtype=np.float32)
            
            # Fill in feature matrix
            for uri in self.nodes_by_type[entity_type]:
                entity_idx = self.entity_to_idx[entity_type][uri]
                for attr, value in self.entities[uri]["attributes"].items():
                    if attr in self.attribute_maps[entity_type]:
                        attr_idx = self.attribute_maps[entity_type][attr]
                        try:
                            feature_matrix[entity_idx, attr_idx] = float(value)
                        except (ValueError, TypeError):
                            # For non-numeric values, use 1.0 to indicate presence
                            feature_matrix[entity_idx, attr_idx] = 1.0
                
                # Handle nodebags - add their attributes to the parent entity
                if uri in self.nodebag_connections:
                    for connection in self.nodebag_connections[uri]:
                        nodebag_uri = connection["nodebag_uri"]
                        nodebag_attrs = self.entities[nodebag_uri]["attributes"]
                        
                        # Handle nodebag attributes with special feature indices
                        for attr, value in nodebag_attrs.items():
                            nodebag_attr_key = f"{connection['nodebag_type']}_{attr}"
                            if nodebag_attr_key in self.attribute_maps[entity_type]:
                                attr_idx = self.attribute_maps[entity_type][nodebag_attr_key]
                                try:
                                    feature_matrix[entity_idx, attr_idx] = float(value)
                                except (ValueError, TypeError):
                                    feature_matrix[entity_idx, attr_idx] = 1.0
            
            # If no attributes, add a dummy feature
            if num_attributes == 0:
                feature_matrix = np.ones((num_entities, 1), dtype=np.float32)
            
            # Normalize features
            if num_entities > 0 and num_attributes > 0:
                # Check if we have any valid features
                if not np.all(feature_matrix == 0):
                    # Replace NaNs with 0
                    feature_matrix = np.nan_to_num(feature_matrix)
                    
                    # Normalize
                    for j in range(feature_matrix.shape[1]):
                        col = feature_matrix[:, j]
                        if np.sum(col != 0) > 0:  # Only normalize non-zero columns
                            mean = np.mean(col[col != 0])
                            std = np.std(col[col != 0])
                            if std > 0:
                                feature_matrix[:, j] = (col - mean) / std
            
            # Convert to tensor
            self.feature_matrices[entity_type] = torch.tensor(feature_matrix, dtype=torch.float)
            print(f"  {entity_type}: {self.feature_matrices[entity_type].shape[0]} nodes, {self.feature_matrices[entity_type].shape[1]} features")
        
        # Create edge indices for each relation type
        print("Creating edge indices...")
        self.edge_indices = {}
        self.edge_types = set()
        
        # Identify all possible relation types
        for entity_data in self.entities.values():
            for rel in entity_data["relationships"]:
                src_type = entity_data["type"]
                dst_uri = rel["target"]
                dst_type = self.entities[dst_uri]["type"]
                rel_type = self.abbreviate_uri(rel["predicate"])
                
                # Create a canonical edge type name: (src_type, rel_type, dst_type)
                edge_type = (src_type, rel_type, dst_type)
                self.edge_types.add(edge_type)
        
        # Create edge indices for each relation type
        for edge_type in self.edge_types:
            src_type, rel_type, dst_type = edge_type
            edge_list = []
            
            # Find all edges of this type
            for uri, entity_data in self.entities.items():
                if entity_data["type"] == src_type:
                    src_idx = self.entity_to_idx[src_type][uri]
                    
                    for rel in entity_data["relationships"]:
                        rel_abbrev = self.abbreviate_uri(rel["predicate"])
                        if rel_abbrev == rel_type:
                            dst_uri = rel["target"]
                            if dst_uri in self.entities and self.entities[dst_uri]["type"] == dst_type:
                                dst_idx = self.entity_to_idx[dst_type][dst_uri]
                                edge_list.append((src_idx, dst_idx))
            
            if edge_list:
                edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
                self.edge_indices[edge_type] = edge_index
                print(f"  {edge_type}: {edge_index.shape[1]} edges")
        
        # Create heterogeneous graph data
        self.create_hetero_data()
        
        return self.hetero_data
            
       
    
    def create_hetero_data(self):
        """Create a HeteroData object for PyTorch Geometric"""
        self.hetero_data = HeteroData()
        
        # Add node features
        for node_type, features in self.feature_matrices.items():
            self.hetero_data[node_type].x = features
            
        # Add edge indices
        for edge_type, edge_index in self.edge_indices.items():
            src_type, rel_type, dst_type = edge_type
            self.hetero_data[src_type, rel_type, dst_type].edge_index = edge_index
            
        print(f"Created heterogeneous graph with {len(self.hetero_data.node_types)} node types and {len(self.hetero_data.edge_types)} edge types")
        return self.hetero_data
    
    def split_edges(self, val_ratio=0.1, test_ratio=0.1):
        """Split edges for training, validation, and testing"""
        train_data = self.hetero_data.clone()
        val_data = self.hetero_data.clone()
        test_data = self.hetero_data.clone()
        
        # For each edge type, split the edges
        for edge_type in self.hetero_data.edge_types:
            edge_index = self.hetero_data[edge_type].edge_index
            num_edges = edge_index.size(1)
            
            if num_edges == 0:
                continue
            
            # Create random permutation
            perm = torch.randperm(num_edges)
            
            # Calculate split sizes
            test_size = int(num_edges * test_ratio)
            val_size = int(num_edges * val_ratio)
            train_size = num_edges - val_size - test_size
            
            # Split indices
            train_indices = perm[:train_size]
            val_indices = perm[train_size:train_size + val_size]
            test_indices = perm[train_size + val_size:]
            
            # Create edge masks
            train_mask = torch.zeros(num_edges, dtype=torch.bool)
            val_mask = torch.zeros(num_edges, dtype=torch.bool)
            test_mask = torch.zeros(num_edges, dtype=torch.bool)
            
            train_mask[train_indices] = True
            val_mask[val_indices] = True
            test_mask[test_indices] = True
            
            # Set edge indices for each split
            train_data[edge_type].edge_index = edge_index[:, train_indices]
            val_data[edge_type].edge_index = edge_index[:, val_indices]
            test_data[edge_type].edge_index = edge_index[:, test_indices]
            
            # Store masks in original data
            self.hetero_data[edge_type].train_mask = train_mask
            self.hetero_data[edge_type].val_mask = val_mask
            self.hetero_data[edge_type].test_mask = test_mask
        
        return train_data, val_data, test_data

In [26]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import HeteroConv, SAGEConv
from sklearn.metrics import roc_auc_score

class HeteroGNN(torch.nn.Module):
    def __init__(self, hetero_data, hidden_channels=64):
        super(HeteroGNN, self).__init__()
        
        # Store node types and edge types
        self.node_types = hetero_data.node_types
        self.edge_types = hetero_data.edge_types
        
        # Initialize node embeddings for each node type
        self.embeddings = torch.nn.ModuleDict()
        for node_type in self.node_types:
            num_features = hetero_data[node_type].x.size(1)
            self.embeddings[node_type] = torch.nn.Linear(num_features, hidden_channels)
        
        # First heterogeneous graph convolution layer
        self.conv1 = HeteroConv({
            edge_type: SAGEConv((-1, -1), hidden_channels)
            for edge_type in self.edge_types
        })
        
        # Second heterogeneous graph convolution layer
        self.conv2 = HeteroConv({
            edge_type: SAGEConv((-1, -1), hidden_channels)
            for edge_type in self.edge_types
        })
        
        # For link prediction
        self.link_predictor = torch.nn.Sequential(
            torch.nn.Linear(hidden_channels * 2, hidden_channels),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_channels, 1)
        )
    
    def forward(self, x_dict, edge_indices_dict):
        # Transform node features
        for node_type in x_dict.keys():
            x_dict[node_type] = self.embeddings[node_type](x_dict[node_type])
            x_dict[node_type] = F.relu(x_dict[node_type])
        
        # First graph convolution
        # Filter out any edge types with empty edge indices
        valid_edge_indices = {k: v for k, v in edge_indices_dict.items() 
                             if v.size(1) > 0}
        
        if valid_edge_indices:
            x_dict = self.conv1(x_dict, valid_edge_indices)
            
            for node_type in x_dict.keys():
                x_dict[node_type] = F.relu(x_dict[node_type])
            
            # Second graph convolution
            x_dict = self.conv2(x_dict, valid_edge_indices)
            
            for node_type in x_dict.keys():
                x_dict[node_type] = F.relu(x_dict[node_type])
        
        return x_dict
    
    def decode(self, z_dict, edge_type, src, dst):
        """Decode embeddings to predict links"""
        src_type, rel_type, dst_type = edge_type
        
        # Get embeddings for source and destination nodes
        src_embeddings = z_dict[src_type][src]
        dst_embeddings = z_dict[dst_type][dst]
        
        # Concatenate embeddings
        x = torch.cat([src_embeddings, dst_embeddings], dim=1)
        
        # Apply MLP to get scores
        return self.link_predictor(x).squeeze()
    
    def decode_all(self, z_dict, edge_type):
        """Decode all possible edges of a given type"""
        src_type, rel_type, dst_type = edge_type
        
        # Get all embeddings
        src_embeddings = z_dict[src_type]
        dst_embeddings = z_dict[dst_type]
        
        # Compute all pairwise scores
        scores = torch.mm(src_embeddings, dst_embeddings.t())
        return scores


# Move these functions outside the class
def train_hetero_gnn(model, optimizer, data):
    """Train the heterogeneous GNN model"""
    model.train()
    optimizer.zero_grad()
    
    # Forward pass - validate input data
    node_features = {node_type: data[node_type].x for node_type in data.node_types}
    edge_indices = {}
    
    # Ensure all edge indices are valid and non-empty
    for edge_type in data.edge_types:
        if edge_type in data.edge_types:
            edge_index = data[edge_type].edge_index
            if edge_index.size(1) > 0:  # Only include non-empty edge indices
                edge_indices[edge_type] = edge_index
    
    # Only proceed with training if there are valid edges
    if not edge_indices:
        print("Warning: No valid edges found for training")
        return 0.0
    
    node_embeddings = model(node_features, edge_indices)
    
    total_loss = 0
    edge_count = 0
    
    # For each edge type, compute link prediction loss
    for edge_type in edge_indices.keys():
        src_type, rel_type, dst_type = edge_type
        edge_index = edge_indices[edge_type]
        
        # Skip if no edges of this type (should not happen due to previous filtering)
        if edge_index.size(1) == 0:
            continue
            
        edge_count += 1
        
        # Generate negative samples
        num_edges = edge_index.size(1)
        num_src_nodes = data[src_type].x.size(0)
        num_dst_nodes = data[dst_type].x.size(0)
        
        # Create random source-destination pairs
        neg_src = torch.randint(0, num_src_nodes, (num_edges,), device=edge_index.device)
        neg_dst = torch.randint(0, num_dst_nodes, (num_edges,), device=edge_index.device)
        
        # Positive samples
        pos_out = model.decode(node_embeddings, edge_type, edge_index[0], edge_index[1])
        
        # Negative samples
        neg_out = model.decode(node_embeddings, edge_type, neg_src, neg_dst)
        
        # Compute loss for this edge type
        edge_loss = F.binary_cross_entropy_with_logits(
            pos_out,
            torch.ones_like(pos_out)
        ) + F.binary_cross_entropy_with_logits(
            neg_out,
            torch.zeros_like(neg_out)
        )
        
        # Add to total loss
        if edge_count == 1:
            total_loss = edge_loss
        else:
            total_loss = total_loss + edge_loss
    
    # Backpropagation
    if edge_count > 0:
        total_loss.backward()
        optimizer.step()
        return total_loss.item() / edge_count  # Return average loss
    else:
        return 0.0

def test_hetero_gnn(model, data):
    """Evaluate the model on the test set"""
    model.eval()
    
    with torch.no_grad():
        # Forward pass - same validation as in training
        node_features = {node_type: data[node_type].x for node_type in data.node_types}
        edge_indices = {}
        
        # Ensure all edge indices are valid and non-empty
        for edge_type in data.edge_types:
            if edge_type in data.edge_types:
                edge_index = data[edge_type].edge_index
                if edge_index.size(1) > 0:  # Only include non-empty edge indices
                    edge_indices[edge_type] = edge_index
        
        # If no valid edges, return empty result
        if not edge_indices:
            print("Warning: No valid edges found for testing")
            return {}
        
        node_embeddings = model(node_features, edge_indices)
        
        auc_scores = {}
        
        # For each edge type, evaluate link prediction
        for edge_type in edge_indices.keys():
            src_type, rel_type, dst_type = edge_type
            edge_index = edge_indices[edge_type]
            
            # Skip if no edges of this type
            if edge_index.size(1) == 0:
                continue
            
            # Generate negative samples
            num_edges = edge_index.size(1)
            num_src_nodes = data[src_type].x.size(0)
            num_dst_nodes = data[dst_type].x.size(0)
            
            # Create random source-destination pairs
            neg_src = torch.randint(0, num_src_nodes, (num_edges,), device=edge_index.device)
            neg_dst = torch.randint(0, num_dst_nodes, (num_edges,), device=edge_index.device)
            
            # Positive samples
            pos_out = model.decode(node_embeddings, edge_type, edge_index[0], edge_index[1])
            
            # Negative samples
            neg_out = model.decode(node_embeddings, edge_type, neg_src, neg_dst)
            
            # Compute AUC score
            y_true = torch.cat([torch.ones_like(pos_out), torch.zeros_like(neg_out)]).cpu().numpy()
            y_score = torch.cat([pos_out, neg_out]).cpu().numpy()
            
            try:
                auc = roc_auc_score(y_true, y_score)
                auc_scores[edge_type] = auc
            except ValueError:
                # Can happen if all samples are from the same class
                auc_scores[edge_type] = 0.5
        
        return auc_scores

In [28]:
import torch
import pandas as pd

def predict_missing_links(model, data, processor, top_k=100, focus_edge_types=None):
    """Predict missing links in the graph"""
    model.eval()
    
    with torch.no_grad():
        # Get node embeddings
        node_embeddings = model(
            {node_type: data[node_type].x for node_type in data.node_types},
            {edge_type: data[edge_type].edge_index for edge_type in data.edge_types}
        )
        
        # Focus on certain edge types if specified
        if focus_edge_types is None:
            edge_types_to_predict = data.edge_types
        else:
            edge_types_to_predict = focus_edge_types
        
        # Collect predictions
        all_predictions = []
        
        for edge_type in edge_types_to_predict:
            src_type, rel_type, dst_type = edge_type
            
            # Get existing edges to filter them out
            existing_edges = set()
            for edge_index in [
                data[edge_type].edge_index,
                data[edge_type].edge_index[:, data[edge_type].train_mask] if hasattr(data[edge_type], 'train_mask') else torch.tensor([[], []], dtype=torch.long),
                data[edge_type].edge_index[:, data[edge_type].val_mask] if hasattr(data[edge_type], 'val_mask') else torch.tensor([[], []], dtype=torch.long),
                data[edge_type].edge_index[:, data[edge_type].test_mask] if hasattr(data[edge_type], 'test_mask') else torch.tensor([[], []], dtype=torch.long)
            ]:
                if edge_index.size(1) > 0:
                    for i in range(edge_index.size(1)):
                        src, dst = edge_index[0, i].item(), edge_index[1, i].item()
                        existing_edges.add((src, dst))
            
            # For a reasonable number of entities, compute all pairwise scores
            num_src_nodes = data[src_type].x.size(0)
            num_dst_nodes = data[dst_type].x.size(0)
            
            # For very large graphs, use sampling instead of all-pairs
            if num_src_nodes * num_dst_nodes > 1_000_000:
                # Sample a subset of source nodes
                sample_size = min(1000, num_src_nodes)
                sampled_src = torch.randperm(num_src_nodes)[:sample_size]
                
                for src_idx in sampled_src:
                    src_idx = src_idx.item()
                    src_emb = node_embeddings[src_type][src_idx:src_idx+1]
                    
                    # Compute scores for all destinations
                    scores = torch.mm(src_emb, node_embeddings[dst_type].t())[0]
                    
                    # Get top-k scores
                    values, indices = torch.topk(scores, min(top_k, scores.size(0)))
                    
                    # Filter out existing edges
                    for i, dst_idx in enumerate(indices):
                        dst_idx = dst_idx.item()
                        if (src_idx, dst_idx) not in existing_edges:
                            score = values[i].item()
                            
                            # Get entity URIs
                            src_uri = processor.idx_to_entity[src_type][src_idx]
                            dst_uri = processor.idx_to_entity[dst_type][dst_idx]
                            
                            # Get entity labels
                            src_label = processor.entities[src_uri].get("abbreviated_uri", src_uri)
                            dst_label = processor.entities[dst_uri].get("abbreviated_uri", dst_uri)
                            
                            all_predictions.append({
                                'source_type': src_type,
                                'source': src_label,
                                'source_uri': src_uri,
                                'relation': rel_type,
                                'target_type': dst_type,
                                'target': dst_label,
                                'target_uri': dst_uri,
                                'score': score
                            })
            else:
                # Compute all pairwise scores
                scores = model.decode_all(node_embeddings, edge_type)
                
                # For each source node, get top-k destinations
                for src_idx in range(num_src_nodes):
                    values, indices = torch.topk(scores[src_idx], min(top_k, scores.size(1)))
                    
                    # Filter out existing edges
                    for i, dst_idx in enumerate(indices):
                        dst_idx = dst_idx.item()
                        if (src_idx, dst_idx) not in existing_edges:
                            score = values[i].item()
                            
                            # Get entity URIs
                            src_uri = processor.idx_to_entity[src_type][src_idx]
                            dst_uri = processor.idx_to_entity[dst_type][dst_idx]
                            
                            # Get entity labels
                            src_label = processor.entities[src_uri].get("abbreviated_uri", src_uri)
                            dst_label = processor.entities[dst_uri].get("abbreviated_uri", dst_uri)
                            
                            all_predictions.append({
                                'source_type': src_type,
                                'source': src_label,
                                'source_uri': src_uri,
                                'relation': rel_type,
                                'target_type': dst_type,
                                'target': dst_label,
                                'target_uri': dst_uri,
                                'score': score
                            })
        
        # Sort all predictions by score
        all_predictions.sort(key=lambda x: x['score'], reverse=True)
        
        # Return top-k overall predictions
        return all_predictions[:top_k]

def analyze_predictions(predictions, processor):
    """Analyze the predictions to help interpret the results"""
    from collections import Counter
    
    # Count predictions by edge type
    edge_type_counts = Counter([(p['source_type'], p['relation'], p['target_type']) for p in predictions])
    
    print("\nPredictions by edge type:")
    for edge_type, count in edge_type_counts.most_common():
        src_type, rel_type, dst_type = edge_type
        print(f"  {src_type} --[{rel_type}]--> {dst_type}: {count} predictions")
    
    # Analyze top predictions for materials
    material_predictions = [p for p in predictions if p['source_type'] == 'nanomaterial' or p['target_type'] == 'nanomaterial']
    
    print("\nTop predictions involving nanomaterials:")
    for i, pred in enumerate(material_predictions[:20]):
        if pred['source_type'] == 'nanomaterial':
            print(f"{i+1}. Nanomaterial {pred['source']} should be connected to {pred['target_type']} {pred['target']} via {pred['relation']} (Score: {pred['score']:.4f})")
        else:
            print(f"{i+1}. {pred['source_type']} {pred['source']} should be connected to nanomaterial {pred['target']} via {pred['relation']} (Score: {pred['score']:.4f})")
    
    # Look for patterns in property predictions
    property_predictions = [p for p in predictions if 'property' in p['source_type'] or 'property' in p['target_type'] or
                           any(prop in p['relation'] for prop in ['purity', 'diameter', 'charge', 'area', 'thickness'])]
    
    print("\nTop property-related predictions:")
    for i, pred in enumerate(property_predictions[:20]):
        print(f"{i+1}. {pred['source_type']}: {pred['source']} --[{pred['relation']}]--> {pred['target_type']}: {pred['target']} (Score: {pred['score']:.4f})")
    
    # Analyze predictions for assay-result relationships
    assay_predictions = [p for p in predictions if p['source_type'] == 'assay' or p['target_type'] == 'assay']
    
    print("\nTop assay-related predictions:")
    for i, pred in enumerate(assay_predictions[:20]):
        print(f"{i+1}. {pred['source_type']}: {pred['source']} --[{pred['relation']}]--> {pred['target_type']}: {pred['target']} (Score: {pred['score']:.4f})")
    
    return {
        'edge_type_counts': edge_type_counts,
        'material_predictions': material_predictions,
        'property_predictions': property_predictions,
        'assay_predictions': assay_predictions
    }

In [29]:
import torch
import os
import pandas as pd
import time
from sklearn.metrics import roc_auc_score

def run_nanomaterial_link_prediction(rdf_file_path, focus_relations=None, output_file="predicted_links.csv"):
    """Run the complete nanomaterial link prediction pipeline"""
    print("=== Nanomaterial Knowledge Base Link Prediction ===")
    print(f"RDF file: {rdf_file_path}")
    
    # Create output directory if it doesn't exist
    os.makedirs(os.path.dirname(output_file) if os.path.dirname(output_file) else '.', exist_ok=True)
    
    # Process RDF data
    start_time = time.time()
    processor = HeterogeneousNKBProcessor(rdf_file_path)
    hetero_data = processor.process_rdf()
    print(f"RDF processing completed in {time.time() - start_time:.2f} seconds")
    
    # Split edges for training/validation/testing
    train_data, val_data, test_data = processor.split_edges()
    
    # Define focus relations if provided
    if focus_relations:
        focus_edge_types = []
        for rel in focus_relations:
            matching_edge_types = []
            for edge_type in hetero_data.edge_types:
                src_type, rel_type, dst_type = edge_type
                if rel_type == rel or rel in rel_type:
                    matching_edge_types.append(edge_type)
            focus_edge_types.extend(matching_edge_types)
        
        print(f"Focusing on {len(focus_edge_types)} edge types")
    else:
        focus_edge_types = None
    
    # Initialize model
    hidden_channels = 64
    model = HeteroGNN(hetero_data, hidden_channels=hidden_channels)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    
    # Check node types and edge types
    print(f"\nNode types: {hetero_data.node_types}")
    edge_count = sum(hetero_data[edge_type].edge_index.size(1) for edge_type in hetero_data.edge_types)
    print(f"Total edges: {edge_count}")
    
    # Verify data splits
    train_edge_count = sum(train_data[edge_type].edge_index.size(1) for edge_type in train_data.edge_types)
    val_edge_count = sum(val_data[edge_type].edge_index.size(1) for edge_type in val_data.edge_types)
    test_edge_count = sum(test_data[edge_type].edge_index.size(1) for edge_type in test_data.edge_types)
    print(f"Train edges: {train_edge_count}")
    print(f"Validation edges: {val_edge_count}")
    print(f"Test edges: {test_edge_count}")
    
    # Train model
    print("\nTraining model...")
    start_time = time.time()
    best_val_auc = 0
    best_model_state = None
    patience = 10
    patience_counter = 0
    
    for epoch in range(100):  # Max 100 epochs
        # Train
        loss = train_hetero_gnn(model, optimizer, train_data)
        
        # Validate
        val_auc_scores = test_hetero_gnn(model, val_data)
        
        # Calculate average AUC (skip if no scores)
        if val_auc_scores:
            val_auc_avg = sum(val_auc_scores.values()) / len(val_auc_scores)
        else:
            val_auc_avg = 0.0
        
        # Print progress
        print(f"Epoch: {epoch:03d}, Loss: {loss:.4f}, Val AUC: {val_auc_avg:.4f}")
        
        # Check for improvement
        if val_auc_avg > best_val_auc:
            best_val_auc = val_auc_avg
            best_model_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping after {epoch} epochs")
                break
    
    print(f"Training completed in {time.time() - start_time:.2f} seconds")
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    # Test final model
    test_auc_scores = test_hetero_gnn(model, test_data)
    
    if test_auc_scores:
        test_auc_avg = sum(test_auc_scores.values()) / len(test_auc_scores)
        print(f"\nTest AUC: {test_auc_avg:.4f}")
        
        # Print AUC for each edge type
        print("\nTest AUC by edge type:")
        for edge_type, auc in sorted(test_auc_scores.items(), key=lambda x: x[1], reverse=True):
            src_type, rel_type, dst_type = edge_type
            print(f"  {src_type} --[{rel_type}]--> {dst_type}: {auc:.4f}")
    else:
        print("\nNo valid test edges found for evaluation")
    
    # Predict missing links
    print("\nPredicting missing links...")
    start_time = time.time()
    predictions = predict_missing_links(model, hetero_data, processor, top_k=1000, focus_edge_types=focus_edge_types)
    print(f"Link prediction completed in {time.time() - start_time:.2f} seconds")
    
    # Save predictions
    predictions_df = pd.DataFrame(predictions)
    predictions_df.to_csv(output_file, index=False)
    print(f"Saved {len(predictions)} predictions to {output_file}")
    
    # Print top predictions
    print("\nTop predicted missing links:")
    for i, pred in enumerate(predictions[:10]):
        print(f"{i+1}. {pred['source_type']}: {pred['source']} --[{pred['relation']}]--> {pred['target_type']}: {pred['target']} (Score: {pred['score']:.4f})")
    
    # Analyze predictions
    analysis = analyze_predictions(predictions, processor)
    
    return predictions, processor, model, analysis

if __name__ == "__main__":
    # Set up focus relations (optional)
    # These are the types of relationships we're most interested in predicting
    focus_relations = [
        'npo:NPO_1808',  # nanomaterial to unknown (core composition)
        'sio:CHEMINF_000568',  # nanomaterial to unknown (DSSTox substance ID)
        'ncit:C42614'  # assay, parameter, result, etc. to unknown (name)
    ]
    
    # Run the pipeline
    rdf_file_path = "./mappings/NKB_RDF_V3.ttl"
    predictions, processor, model, analysis = run_nanomaterial_link_prediction(
        rdf_file_path, 
        focus_relations=focus_relations,
        output_file="results/nkb_predicted_links.csv"
    )

=== Nanomaterial Knowledge Base Link Prediction ===
RDF file: ./mappings/NKB_RDF_V3.ttl
Loading RDF file: ./mappings/NKB_RDF_V3.ttl
Loaded 1369675 triples

Sample triples (10):
Subject: http://example.org/parameters/38214
Predicate: ncit:C42614
Object: npo:NPO_1830
---
Subject: http://example.org/parameters/70617
Predicate: ncit:C25704
Object: nan...
---
Subject: http://example.org/parameters/15210
Predicate: ncit:C25704
Object: nan...
---
Subject: http://example.org/assay/14830
Predicate: ncit:C42614
Object: npo:NPO_1469
---
Subject: ne35fd35e09e7488a82a87a9e28ba9f15b118305
Predicate: dcterms:identifier
Object: 52...
---
Subject: http://example.org/parameters/55378
Predicate: ncit:C25704
Object: nan...
---
Subject: ne35fd35e09e7488a82a87a9e28ba9f15b127969
Predicate: dcterms:identifier
Object: 665...
---
Subject: http://example.org/assay/5297
Predicate: ncit:C42614
Object: enm:ENM_8000275
---
Subject: http://example.org/parameters/34524
Predicate: rdf:type
Object: npo:NPO_1680
---
Subj

/Users/pranavsingh/anaconda3/lib/python3.11/site-packages/torch_geometric/nn/conv/hetero_conv.py:76: UserWarning: There exist node types ({'parameter', 'publication', 'nanomaterial', 'result', 'medium', 'other', 'additive', 'contaminant', 'assay', 'functional_group'}) whose representations do not get updated during message passing as they do not occur as destination type in any edge type. This may lead to unexpected behavior.
  warnings.warn(


AttributeError: 'NoneType' object has no attribute 'dim'

In [34]:
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
import time

def simplified_link_prediction(processor, top_k=100, focus_edge_types=None):
    """
    A simplified approach to link prediction that doesn't rely on GNN message passing.
    This handles feature matrices of different dimensions.
    
    Args:
        processor: The RDF processor instance with entity data
        top_k: Number of top predictions to return
        focus_edge_types: Optional list of edge types to focus on
        
    Returns:
        List of predicted links
    """
    print("Starting simplified link prediction...")
    start_time = time.time()
    
    # Collect entities by type
    entities_by_type = {}
    for entity_uri, entity_data in processor.entities.items():
        entity_type = entity_data["type"]
        if entity_type not in entities_by_type:
            entities_by_type[entity_type] = []
        entities_by_type[entity_type].append(entity_uri)
    
    # Identify existing edges to avoid predicting them
    existing_edges = set()
    for entity_uri, entity_data in processor.entities.items():
        for rel in entity_data.get("relationships", []):
            src_uri = entity_uri
            src_type = entity_data["type"]
            dst_uri = rel["target"]
            
            # Skip if target doesn't exist in our entities
            if dst_uri not in processor.entities:
                continue
                
            dst_type = processor.entities[dst_uri]["type"]
            rel_type = rel["abbreviated_predicate"]
            
            # Create a unique identifier for this edge
            edge_key = (src_uri, rel_type, dst_uri)
            existing_edges.add(edge_key)
    
    print(f"Found {len(existing_edges)} existing edges to exclude")
    
    # Determine edge types to predict
    if focus_edge_types is None:
        # Use specific source-destination type pairs that make sense
        edge_types_to_predict = []
        for src_type in entities_by_type:
            for dst_type in entities_by_type:
                # For simplicity, use a generic relation type
                # You can refine this based on domain knowledge
                edge_types_to_predict.append((src_type, "potential_relationship", dst_type))
    else:
        edge_types_to_predict = focus_edge_types
    
    print(f"Will predict links for {len(edge_types_to_predict)} edge types")
    
    # Collect predictions
    all_predictions = []
    
    # For each edge type
    for edge_type in edge_types_to_predict:
        src_type, rel_type, dst_type = edge_type
        
        # Skip if we don't have entities of these types
        if src_type not in entities_by_type or dst_type not in entities_by_type:
            continue
        
        print(f"Processing edge type: {src_type} --[{rel_type}]--> {dst_type}")
        
        # Extract the attribute data for each entity
        src_entities = [processor.entities[uri] for uri in entities_by_type[src_type]]
        dst_entities = [processor.entities[uri] for uri in entities_by_type[dst_type]]
        
        # Calculate all pairwise similarities based on attribute overlap
        for src_idx, src_entity in enumerate(src_entities):
            src_uri = src_entity["uri"]
            
            # Get source attributes
            src_attrs = set()
            for attr, value in src_entity.get("attributes", {}).items():
                if value and value != "nan":
                    src_attrs.add(f"{attr}:{value}")
            
            # For small graphs, process all destinations
            # For large graphs, sample destinations
            dst_indices = list(range(len(dst_entities)))
            if len(dst_indices) > 1000:
                # Sample 1000 random destinations for efficiency
                dst_indices = np.random.choice(dst_indices, 1000, replace=False)
            
            for dst_idx in dst_indices:
                dst_entity = dst_entities[dst_idx]
                dst_uri = dst_entity["uri"]
                
                # Skip if this edge already exists
                edge_key = (src_uri, rel_type, dst_uri)
                if edge_key in existing_edges:
                    continue
                
                # Skip self-loops
                if src_uri == dst_uri:
                    continue
                
                # Get destination attributes
                dst_attrs = set()
                for attr, value in dst_entity.get("attributes", {}).items():
                    if value and value != "nan":
                        dst_attrs.add(f"{attr}:{value}")
                
                # Calculate similarity based on attribute overlap
                # We use Jaccard similarity: |A ∩ B| / |A ∪ B|
                if src_attrs or dst_attrs:
                    intersection = len(src_attrs.intersection(dst_attrs))
                    union = len(src_attrs.union(dst_attrs))
                    
                    if union > 0:
                        similarity = intersection / union
                    else:
                        similarity = 0
                else:
                    similarity = 0
                
                # Only keep predictions with some similarity
                if similarity > 0:
                    # Get entity labels
                    src_label = src_entity.get("abbreviated_uri", src_uri)
                    dst_label = dst_entity.get("abbreviated_uri", dst_uri)
                    
                    # Add prediction
                    all_predictions.append({
                        'source_type': src_type,
                        'source': src_label,
                        'source_uri': src_uri,
                        'relation': rel_type,
                        'target_type': dst_type,
                        'target': dst_label,
                        'target_uri': dst_uri,
                        'score': float(similarity)
                    })
    
    # Sort by score
    all_predictions.sort(key=lambda x: x['score'], reverse=True)
    
    # Keep only top-k overall
    all_predictions = all_predictions[:top_k]
    
    print(f"Link prediction completed in {time.time() - start_time:.2f} seconds")
    
    return all_predictions


def similarity_based_predictions_for_materials(processor, top_k=100):
    """
    Predict links specifically for nanomaterials using property similarity
    """
    print("Starting nanomaterial-focused link prediction...")
    start_time = time.time()
    
    # Define the nanomaterial type
    material_type = 'nanomaterial'
    
    # Get all nanomaterials
    materials = []
    for uri, entity in processor.entities.items():
        if entity["type"] == material_type:
            materials.append(entity)
    
    print(f"Found {len(materials)} nanomaterials")
    
    # Get existing links for nanomaterials
    existing_edges = set()
    for material in materials:
        for rel in material.get("relationships", []):
            src_uri = material["uri"]
            dst_uri = rel["target"]
            rel_type = rel["abbreviated_predicate"]
            edge_key = (src_uri, rel_type, dst_uri)
            existing_edges.add(edge_key)
    
    # Get all property types that might be associated with nanomaterials
    rel_types = set()
    for material in materials:
        for rel in material.get("relationships", []):
            rel_types.add(rel["abbreviated_predicate"])
    
    print(f"Found {len(rel_types)} relation types for nanomaterials")
    
    # Collect property values for each material
    material_properties = {}
    for material in materials:
        material_uri = material["uri"]
        material_properties[material_uri] = {}
        
        # Add direct attributes
        for attr, value in material.get("attributes", {}).items():
            if value and value != "nan":
                material_properties[material_uri][attr] = value
        
        # Add relationships
        for rel in material.get("relationships", []):
            rel_type = rel["abbreviated_predicate"]
            target_uri = rel["target"]
            
            # If we already have this property, make it a list
            if rel_type in material_properties[material_uri]:
                if isinstance(material_properties[material_uri][rel_type], list):
                    material_properties[material_uri][rel_type].append(target_uri)
                else:
                    material_properties[material_uri][rel_type] = [
                        material_properties[material_uri][rel_type], target_uri
                    ]
            else:
                material_properties[material_uri][rel_type] = target_uri
    
    # Calculate similarities between materials
    predictions = []
    
    for i, material1 in enumerate(materials):
        material1_uri = material1["uri"]
        props1 = material_properties.get(material1_uri, {})
        
        for j, material2 in enumerate(materials):
            # Skip same material
            if i == j:
                continue
                
            material2_uri = material2["uri"]
            props2 = material_properties.get(material2_uri, {})
            
            # Calculate property similarity
            shared_props = set(props1.keys()).intersection(set(props2.keys()))
            all_props = set(props1.keys()).union(set(props2.keys()))
            
            # Jaccard similarity for property keys
            if all_props:
                key_similarity = len(shared_props) / len(all_props)
            else:
                key_similarity = 0
            
            # Calculate similarity of property values
            value_similarity = 0
            if shared_props:
                matching_values = 0
                for prop in shared_props:
                    val1 = props1[prop]
                    val2 = props2[prop]
                    
                    # Handle different data types
                    if isinstance(val1, list) and isinstance(val2, list):
                        # Calculate overlap
                        common = set(val1).intersection(set(val2))
                        total = set(val1).union(set(val2))
                        if total:
                            matching_values += len(common) / len(total)
                    elif val1 == val2:
                        matching_values += 1
                
                value_similarity = matching_values / len(shared_props)
            
            # Combined similarity score
            similarity = (key_similarity + value_similarity) / 2
            
            # Skip zero similarity
            if similarity <= 0:
                continue
            
            # For similar materials, suggest potential missing relationships
            if similarity > 0.1:  # Threshold for similarity
                # Find relationships that material2 has but material1 doesn't
                for rel_type in props2:
                    if rel_type not in props1:
                        # This is a relationship type material1 is missing
                        
                        # If it's a list of values, suggest each one
                        values_to_suggest = props2[rel_type]
                        if not isinstance(values_to_suggest, list):
                            values_to_suggest = [values_to_suggest]
                        
                        for target_uri in values_to_suggest:
                            # Skip if this relation already exists
                            edge_key = (material1_uri, rel_type, target_uri)
                            if edge_key in existing_edges:
                                continue
                            
                            # Skip if target_uri doesn't exist in the entity list
                            if target_uri not in processor.entities:
                                continue
                                
                            target_type = processor.entities[target_uri]["type"]
                            
                            # Add prediction
                            predictions.append({
                                'source_type': material_type,
                                'source': material1.get("abbreviated_uri", material1_uri),
                                'source_uri': material1_uri,
                                'relation': rel_type,
                                'target_type': target_type,
                                'target': processor.entities[target_uri].get("abbreviated_uri", target_uri),
                                'target_uri': target_uri,
                                'score': float(similarity),
                                'evidence': f"Similar material {material2.get('abbreviated_uri', material2_uri)} has this relationship"
                            })
    
    # Sort by score
    predictions.sort(key=lambda x: x['score'], reverse=True)
    
    # Keep only top-k
    predictions = predictions[:top_k]
    
    print(f"Material similarity prediction completed in {time.time() - start_time:.2f} seconds")
    
    return predictions


def suggested_edge_types_for_nanomaterials():
    """
    Returns suggested edge types to focus on for nanomaterials
    """
    return [
        ('nanomaterial', 'npo:NPO_1808', 'unknown'),  # Core composition
        ('nanomaterial', 'sio:CHEMINF_000568', 'unknown'),  # DSSTox ID
        ('nanomaterial', 'npo:NPO_1823', 'unknown'),  # Coating composition
        ('nanomaterial', 'npo:NPO_1824', 'unknown'),  # Shell composition
        ('nanomaterial', 'enm:ENM_8000049', 'unknown'),  # Shape in medium
        ('nanomaterial', 'ncit:C25365', 'unknown'),  # Description
    ]


def analyze_simplified_predictions(predictions, processor):
    """
    Analyze the simplified predictions to help interpret the results
    """
    from collections import Counter
    
    # Count predictions by edge type
    edge_type_counts = Counter([(p['source_type'], p['relation'], p['target_type']) for p in predictions])
    
    print("\nPredictions by edge type:")
    for edge_type, count in edge_type_counts.most_common():
        src_type, rel_type, dst_type = edge_type
        print(f"  {src_type} --[{rel_type}]--> {dst_type}: {count} predictions")
    
    # Analyze top predictions for materials
    material_predictions = [p for p in predictions if p['source_type'] == 'nanomaterial' or p['target_type'] == 'nanomaterial']
    
    print("\nTop predictions involving nanomaterials:")
    for i, pred in enumerate(material_predictions[:10]):
        if pred['source_type'] == 'nanomaterial':
            print(f"{i+1}. Nanomaterial {pred['source']} should be connected to {pred['target_type']} {pred['target']} via {pred['relation']} (Score: {pred['score']:.4f})")
        else:
            print(f"{i+1}. {pred['source_type']} {pred['source']} should be connected to nanomaterial {pred['target']} via {pred['relation']} (Score: {pred['score']:.4f})")
    
    # Look for patterns in property predictions
    property_predictions = [p for p in predictions if 'property' in p['source_type'] or 'property' in p['target_type']]
    
    if property_predictions:
        print("\nTop property-related predictions:")
        for i, pred in enumerate(property_predictions[:5]):
            print(f"{i+1}. {pred['source_type']}: {pred['source']} --[{pred['relation']}]--> {pred['target_type']}: {pred['target']} (Score: {pred['score']:.4f})")
    
    # Get unique materials involved in predictions
    material_uris = set()
    for pred in material_predictions:
        if pred['source_type'] == 'nanomaterial':
            material_uris.add(pred['source_uri'])
        if pred['target_type'] == 'nanomaterial':
            material_uris.add(pred['target_uri'])
    
    print(f"\nNumber of unique nanomaterials in predictions: {len(material_uris)}")
    
    return {
        'edge_type_counts': edge_type_counts,
        'material_predictions': material_predictions,
        'property_predictions': property_predictions,
        'material_uris': material_uris
    }


def run_simplified_link_prediction(processor, focus_relations=None, output_file="predicted_links.csv", top_k=100):
    """
    Run the simplified link prediction pipeline
    """
    print("=== Running Simplified Link Prediction for NanoKnowBase ===")
    
    # Try two different approaches and combine results
    print("\n[1/2] Using attribute-based similarity...")
    predictions1 = simplified_link_prediction(
        processor=processor,
        top_k=top_k,
        focus_edge_types=None  # Use default edge types
    )
    
    print("\n[2/2] Using nanomaterial-specific similarity...")
    predictions2 = similarity_based_predictions_for_materials(
        processor=processor,
        top_k=top_k
    )
    
    # Combine predictions
    combined_predictions = []
    seen_edges = set()
    
    # Add predictions from first method
    for pred in predictions1:
        edge_key = (pred['source_uri'], pred['relation'], pred['target_uri'])
        if edge_key not in seen_edges:
            combined_predictions.append(pred)
            seen_edges.add(edge_key)
    
    # Add predictions from second method
    for pred in predictions2:
        edge_key = (pred['source_uri'], pred['relation'], pred['target_uri'])
        if edge_key not in seen_edges:
            combined_predictions.append(pred)
            seen_edges.add(edge_key)
    
    # Sort by score
    combined_predictions.sort(key=lambda x: x['score'], reverse=True)
    
    # Limit to top-k
    combined_predictions = combined_predictions[:top_k]
    
    # Save predictions to CSV
    if combined_predictions:
        predictions_df = pd.DataFrame(combined_predictions)
        predictions_df.to_csv(output_file, index=False)
        print(f"\nSaved {len(combined_predictions)} predictions to {output_file}")
        
        # Print top predictions
        print("\nTop predicted missing links:")
        for i, pred in enumerate(combined_predictions[:10]):
            print(f"{i+1}. {pred['source_type']}: {pred['source']} --[{pred['relation']}]--> {pred['target_type']}: {pred['target']} (Score: {pred['score']:.4f})")
        
        # Analyze predictions
        analysis = analyze_simplified_predictions(combined_predictions, processor)
        
        return combined_predictions, analysis
    else:
        print("No predictions generated.")
        return [], None

In [35]:
processor = HeterogeneousNKBProcessor(rdf_file_path)
hetero_data = processor.process_rdf()


# Define focus relations if needed
focus_relations = [
    'npo:NPO_1808',        # Core composition
    'sio:CHEMINF_000568',  # DSSTox substance ID
    'ncit:C42614'          # Name property
]

# Run simplified link prediction
predictions, analysis = run_simplified_link_prediction(
    processor=processor,
    focus_relations=focus_relations,
    output_file="results/nkb_predicted_links.csv",
    top_k=100  # Number of predictions to generate
)

Loading RDF file: ./mappings/NKB_RDF_V3.ttl
Loaded 1369675 triples

Sample triples (10):
Subject: http://example.org/parameters/38214
Predicate: ncit:C42614
Object: npo:NPO_1830
---
Subject: http://example.org/parameters/10269
Predicate: obo:OBI_0000070
Object: nb1e663512e09472c9a3026f807fc4f17b50956
---
Subject: http://example.org/parameters/70617
Predicate: ncit:C25704
Object: nan...
---
Subject: http://example.org/parameters/15210
Predicate: ncit:C25704
Object: nan...
---
Subject: nb1e663512e09472c9a3026f807fc4f17b124820
Predicate: obo:OBI_0002110
Object: 10.1016/j.scitotenv.2016.11.029...
---
Subject: http://example.org/parameters/75326
Predicate: obo:OBI_0000070
Object: nb1e663512e09472c9a3026f807fc4f17b123241
---
Subject: http://example.org/assay/14830
Predicate: ncit:C42614
Object: npo:NPO_1469
---
Subject: http://example.org/assay/14658
Predicate: ncit:C93400
Object: nb1e663512e09472c9a3026f807fc4f17b10659
---
Subject: http://example.org/parameters/55378
Predicate: ncit:C25704


In [ ]:
import rdflib
from rdflib import Graph, URIRef, Namespace
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx

def explore_rdf_structure(rdf_file_path, format="turtle"):
    """Explore and visualize the structure of an RDF file"""
    print(f"Loading RDF from {rdf_file_path}...")
    g = Graph()
    
    # Set up namespaces
    namespaces = {
        'ncit': Namespace('http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#'),
        'obo': Namespace('http://purl.obolibrary.org/obo/'),
        'dcterms': Namespace('http://purl.org/dc/terms/'),
        'npo': Namespace('http://purl.bioontology.org/ontology/npo#'),
        'dc': Namespace('http://purl.org/dc/elements/1.1/'),
        'enm': Namespace('http://purl.enanomapper.org/onto/'),
        'sio': Namespace('http://semanticscience.org/resource/'),
        'rdf': Namespace('http://www.w3.org/1999/02/22-rdf-syntax-ns#'),
        'rdfs': Namespace('http://www.w3.org/2000/01/rdf-schema#')
    }
    
    # Bind namespaces
    for prefix, namespace in namespaces.items():
        g.bind(prefix, namespace)
    
    # Parse RDF file
    g.parse(rdf_file_path, format=format)
    print(f"Loaded {len(g)} triples")
    
    # Helper function to abbreviate URIs
    def abbreviate_uri(uri):
        for namespace_uri, prefix in [(str(ns), p) for p, ns in namespaces.items()]:
            if uri.startswith(namespace_uri):
                return f"{prefix}:{uri[len(namespace_uri):]}"
        return uri
    
    # 1. Basic Statistics
    print("\n==== Basic Graph Statistics ====")
    subjects = set(g.subjects())
    predicates = set(g.predicates())
    objects = set(g.objects())
    
    print(f"Unique subjects: {len(subjects)}")
    print(f"Unique predicates: {len(predicates)}")
    print(f"Unique objects: {len(objects)}")
    
    # Count entity types
    subject_types = Counter([type(s).__name__ for s in subjects])
    object_types = Counter([type(o).__name__ for o in objects])
    
    print("\nSubject types:")
    for t, count in subject_types.items():
        print(f"  {t}: {count}")
    
    print("\nObject types:")
    for t, count in object_types.items():
        print(f"  {t}: {count}")
    
    # 2. Examine Predicates (Relations)
    print("\n==== Predicates/Relations ====")
    predicate_counts = Counter()
    
    for p in predicates:
        count = len(list(g.triples((None, p, None))))
        predicate_counts[str(p)] = count
    
    # Display top 10 most common predicates
    print("Top 10 most common predicates:")
    for p, count in predicate_counts.most_common(10):
        print(f"  {abbreviate_uri(p)} ({count} occurrences)")
    
    # 3. Count RDF types
    print("\n==== Entity Types ====")
    rdf_type = namespaces['rdf']['type']
    type_counts = Counter()
    
    for s, p, o in g.triples((None, rdf_type, None)):
        if isinstance(o, URIRef):
            type_counts[str(o)] += 1
    
    print("Top 10 most common entity types:")
    for t, count in type_counts.most_common(10):
        print(f"  {abbreviate_uri(t)}: {count} entities")
    
    # 4. Visualize node type distribution
    plt.figure(figsize=(10, 6))
    types = [abbreviate_uri(t) for t, _ in type_counts.most_common(10)]
    counts = [c for _, c in type_counts.most_common(10)]
    plt.bar(types, counts)
    plt.xticks(rotation=45, ha='right')
    plt.title('Top 10 Entity Types')
    plt.tight_layout()
    plt.savefig('entity_types.png')
    print("\nSaved entity type distribution to 'entity_types.png'")
    
    # 5. Examine Nodebags
    print("\n==== Nodebag Analysis ====")
    nodebag_prefixes = [
        'synthesis',
        'diameter',
        'length',
        'thickness',
        'surface_area',
        'size_distribution',
        'hydrodynamic',
        'surface_charge',
        'purity'
    ]
    
    # Count attributes in each nodebag type
    nodebag_attributes = defaultdict(set)
    for s, p, o in g:
        s_str = str(s)
        p_str = str(p)
        
        for prefix in nodebag_prefixes:
            if prefix in s_str.lower():
                nodebag_attributes[prefix].add(p_str)
    
    print("Attributes by nodebag type:")
    for nodebag_type, attrs in nodebag_attributes.items():
        print(f"  {nodebag_type}: {len(attrs)} unique attributes")
        for attr in sorted(list(attrs))[:5]:  # Show first 5 attributes
            print(f"    - {abbreviate_uri(attr)}")
        if len(attrs) > 5:
            print(f"    - ... and {len(attrs) - 5} more")
    
    # 6. Create a sample visualization
    print("\n==== Creating Sample Graph Visualization ====")
    # Create a small sample networkx graph for visualization
    G = nx.DiGraph()
    
    # Find a material entity to use as a starting point
    material_uri = None
    for s, p, o in g.triples((None, rdf_type, namespaces['npo']['NPO_199'])):
        material_uri = s
        break
    
    if material_uri:
        # Add material node
        material_label = abbreviate_uri(str(material_uri))
        G.add_node(material_label, type='nanomaterial')
        
        # Add direct connections
        for _, p, o in g.triples((material_uri, None, None)):
            if isinstance(o, URIRef):
                target_label = abbreviate_uri(str(o))
                G.add_node(target_label, type='target')
                G.add_edge(material_label, target_label, label=abbreviate_uri(str(p)))
        
        # Visualize
        plt.figure(figsize=(12, 8))
        pos = nx.spring_layout(G)
        
        # Draw nodes
        nx.draw_networkx_nodes(G, pos, 
                               node_color=['lightblue' if d['type'] == 'nanomaterial' else 'lightgreen' 
                                          for n, d in G.nodes(data=True)],
                               node_size=500)
        
        # Draw edges
        nx.draw_networkx_edges(G, pos, alpha=0.5)
        
        # Draw labels
        nx.draw_networkx_labels(G, pos, font_size=8)
        
        # Draw edge labels (only for a subset to avoid clutter)
        edge_labels = {(u, v): d['label'] for u, v, d in G.edges(data=True)}
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=6)
        
        plt.title(f"Connections for a sample nanomaterial")
        plt.axis('off')
        plt.tight_layout()
        plt.savefig('sample_graph.png', dpi=300)
        print("Saved sample graph visualization to 'sample_graph.png'")
    
    return g, predicate_counts, type_counts, nodebag_attributes

if __name__ == "__main__":
    rdf_file_path = "./mappings/NKB_RDF_V3.ttl"  # Replace with your RDF file path
    g, predicate_counts, type_counts, nodebag_attributes = explore_rdf_structure(rdf_file_path)
    
    # Additional interactive exploration could be added here
    print("\nExploration complete. You can now interactively explore the RDF graph.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

def visualize_predictions(predictions_file):
    """Visualize the link predictions"""
    # Load predictions
    predictions = pd.read_csv(predictions_file)
    
    print(f"Loaded {len(predictions)} predictions from {predictions_file}")
    
    # 1. Distribution of prediction scores
    plt.figure(figsize=(10, 6))
    sns.histplot(predictions['score'], bins=30, kde=True)
    plt.title('Distribution of Prediction Scores')
    plt.xlabel('Score')
    plt.ylabel('Count')
    plt.savefig('prediction_scores.png')
    
    # 2. Top predictions by node type
    plt.figure(figsize=(12, 8))
    
    # Count predictions by source and target type
    edge_counts = predictions.groupby(['source_type', 'target_type']).size().reset_index(name='count')
    edge_counts = edge_counts.sort_values('count', ascending=False)
    
    # Plot top 15
    sns.barplot(data=edge_counts.head(15), x='count', y=edge_counts.head(15).apply(lambda x: f"{x['source_type']} → {x['target_type']}", axis=1))
    plt.title('Top 15 Predicted Edge Types')
    plt.xlabel('Count')
    plt.tight_layout()
    plt.savefig('top_edge_types.png')
    
    # 3. Network visualization of top predictions
    # Create a graph of top 50 predictions
    G = nx.DiGraph()
    
    # Add top predictions
    for _, row in predictions.head(50).iterrows():
        # Create node labels
        source_label = f"{row['source_type']}: {row['source']}"
        target_label = f"{row['target_type']}: {row['target']}"
        
        # Add nodes with types
        G.add_node(source_label, node_type=row['source_type'])
        G.add_node(target_label, node_type=row['target_type'])
        
        # Add edge with score
        G.add_edge(source_label, target_label, relation=row['relation'], score=row['score'])
    
    # Visualize
    plt.figure(figsize=(15, 12))
    
    # Use different colors for different node types
    node_types = set(nx.get_node_attributes(G, 'node_type').values())
    color_map = {}
    for i, nt in enumerate(node_types):
        color_map[nt] = plt.cm.tab10(i / len(node_types))
    
    node_colors = [color_map[G.nodes[n]['node_type']] for n in G.nodes]
    
    # Layout
    pos = nx.spring_layout(G, k=0.3, iterations=50)
    
    # Draw nodes
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=300, alpha=0.8)
    
    # Draw edges with width based on score
    edge_widths = [G[u][v]['score'] * 5 for u, v in G.edges]
    nx.draw_networkx_edges(G, pos, width=edge_widths, alpha=0.6, edge_color='gray')
    
    # Draw labels with smaller font
    nx.draw_networkx_labels(G, pos, font_size=8)
    
    # Add a legend for node types
    legend_elements = [plt.Line2D([0], [0], marker='o', color='w', 
                                 label=nt, markerfacecolor=color_map[nt], markersize=10)
                      for nt in node_types]
    plt.legend(handles=legend_elements, title="Node Types", loc='upper right')
    
    plt.title('Top 50 Predicted Missing Links')
    plt.axis('off')
    plt.tight_layout()
    plt.savefig('prediction_network.png', dpi=300)
    
    print("Visualizations saved as prediction_scores.png, top_edge_types.png, and prediction_network.png")
    
    return predictions

if __name__ == "__main__":
    predictions_file = "results/nkb_predicted_links.csv"  # Replace with your predictions file
    predictions = visualize_predictions(predictions_file)